# Cardiovascular arousal response strength is related to obstructive sleep apnea severity, impaired vigilance, and sleep depth
Code by Mieli Luukinen, licensed with the Academic Free License v3.0.

There are only few comments, with the main purpose of describing what needs to be done to replicate our results with your own data, rather than to explain what the code does. I have attempted to remove all code that isn't relevant to the final publication, but some remnants may still be there. There is also a slight risk that in that process I have removed something that is relevant, but I don't think so.

## Preparations

In [ ]:
%matplotlib widget

In [ ]:
# These supposedly limit the use of CPU cores, which may be useful on a shared server.

import os

NUM_THREADS = "16"

os.environ["OMP_NUM_THREADS"] = NUM_THREADS
os.environ["OPENBLAS_NUM_THREADS"] = NUM_THREADS
os.environ["MKL_NUM_THREADS"] = NUM_THREADS
os.environ["VECLIB_MAXIMUM_THREADS"] = NUM_THREADS
os.environ["NUMEXPR_NUM_THREADS"] = NUM_THREADS

In [ ]:
from os import listdir
from os.path import isfile, join
import csv
import datetime
import itertools
import json
import pickle
from collections import Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec
import scipy.signal, scipy.stats, scipy.optimize, scipy.ndimage
from scipy.integrate import trapezoid
from IPython.display import clear_output
from tqdm.notebook import tqdm, trange
from tqdm.contrib.concurrent import process_map
from tqdm import tqdm as tqdm_console
from statsmodels.stats.descriptivestats import sign_test
from statsmodels.discrete.discrete_model import Logit
from statsmodels.regression.mixed_linear_model import MixedLM
import openpyxl
import pandas
from tabulate import tabulate, SEPARATING_LINE
from docx import Document

from utils import *
from fit_analysis import *

In [ ]:
figfont8 = {'fontname':'Times New Roman', 'fontsize':8}
figfont10 = {'fontname':'Times New Roman', 'fontsize':10}
matplotlib.rcParams['mathtext.fontset'] = 'stix'

In [ ]:
# So, lots of data in different folders. You probably need to do some tweaking to use the code with your own data. Sadly, I don't have exact specifications for the formats our data are in, so you are
# mostly on your own. The main thing is that you need scored polysomnograms and psychomotor vigilance task data.

# Here the path for photoplethysmogram data is commented out due to some changes in our directories after I had extracted what I needed from them and saved it elsewhere. You should probably change that.
# EDIT: in the revision, I am again analysing the photoplethysmogram data for normalization and to assess the goodness of fit. For simplicity and saving RAM I will not re-do the whole thing where PPG
# data would be stored for every arousal. Thus, the comments about the need to rewrite some PPG handling code if you replicate the study are still valid.

event_path = '/wrk/luukinen/data/event_files'
header_path = '/wrk/luukinen/data/headers/'
hypno_path = '/wrk/luukinen/data/hypnograms'
pvt_path = '/wrk/luukinen/data/PVT_results'
headerstart = 'header_Compumedics ProFusion PSG - '
headerend = '.json'
eventstart = 'events_Compumedics ProFusion PSG - '
eventend = '.TXT'
hypnostart = 'hypnogram_Compumedics ProFusion PSG - '
hypnoend = '.TXT'
pvtstart = 'pvt_'
pvtend = '.xlsx'
demographics_path = '/wrk/luukinen/patientdata.xlsx'

pleth_path = '/wrk/luukinen/data/pa/pa_extracted_fixed/psg_2015_2017/'
plethend = 'Pleth_64Hz/data.npy'

In [ ]:
event_files = [f for f in tqdm(listdir(event_path)) if isfile(join(event_path, f))]
header_files = [f for f in tqdm(listdir(header_path)) if isfile(join(header_path, f))]
hypnograms = [f for f in tqdm(listdir(hypno_path)) if isfile(join(hypno_path, f))]
pleth_files = [f for f in tqdm(listdir(pleth_path)) if isfile(join(pleth_path, f, plethend))]
pvt_files = [f for f in tqdm(listdir(pvt_path)) if isfile(join(pvt_path, f))]
# A somewhat slow way of dealing with the fact that edf_files has a subfolder.
# It would be faster to sort the lists and discard the last one in that one.
# However, this is more robust if someone changes  the files.

# Those last comments are old, as seen from the fact the code no longer uses data in EDF format.

In [ ]:
fit_path = '/wrk/luukinen/results/20220426/'
patients_fit = [f for f in tqdm(listdir(fit_path)) if isfile(join(fit_path, f))]
fit_path_spont = '/wrk/luukinen/results/20221101_spont/'
patients_fit_spont = [f for f in tqdm(listdir(fit_path_spont)) if isfile(join(fit_path_spont, f))]

In [ ]:
# Matching files. The code is ugly due to some varying file name conventions etc.
files = {hdr.split()[4][1:]: [header_path + '/' + hdr,
 event_path + '/' + eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend,
 hypno_path + '/' + hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend,
 pleth_path + hdr.split()[4][1:] + '/' + plethend,
 pvt_path + '/' + pvtstart + hdr.split()[4][1:] + pvtend]
 for hdr in tqdm(header_files)
 if (eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend) in event_files
 and (hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend) in hypnograms
 and hdr.split()[4][1:] in pleth_files
 and pvtstart + hdr.split()[4][1:] + pvtend in pvt_files
 and (hdr.split()[4][1:] in patients_fit or hdr.split()[4][1:] in patients_fit_spont)
}

In [ ]:
patient_ids = sorted(list(files))

In [ ]:
wb = openpyxl.load_workbook('/wrk/luukinen/patientdata.xlsx', data_only=True)
ws = wb.active
demographics = []
for row in ws.values:
    demographics.append([])
    for value in row:
        demographics[-1].append(value)

In [ ]:
demographics_by_patient = {str(demographics[i][0]): demographics[i] for i in range(len(demographics)) if isinstance(demographics[i][0], int) and str(demographics[i][0]) in patient_ids}

In [ ]:
len([v for v in demographics_by_patient.values() if isinstance(v[3], int) or isinstance(v[3], float)])
exclude_no_demographics = []
for patient in patient_ids:
    v = demographics_by_patient[patient]
    if not (v[1] == 0 or v[1] == 1) or not (isinstance(v[2], int) or isinstance(v[2], float)) or not (isinstance(v[3], int) or isinstance(v[3], float)) or v[3] == 0:
        exclude_no_demographics.append(patient)
print(len(exclude_no_demographics))

# NOTE: This is untrue, check the number from previous paper!
# ANOTHER NOTE, MUCH LATER: This number is supposed to tell how many subjects were excluded for not having demographic data. I am no longer exactly sure why it's wrong.

In [ ]:
pvt_rts_by_patient = {}
for patient in tqdm(patient_ids):
    pvt_rts_by_patient[patient] = pandas.read_excel(files[patient][4])['rt'].to_numpy()

In [ ]:
# This data is desaturation severity (DesSev) for each subject, calculated elsewhere.

wb = openpyxl.load_workbook('/wrk/luukinen/DesSev_table.xlsx', data_only=True)
ws = wb.active
dessevs_ = []
for row in ws.values:
    dessevs_.append([])
    for value in row:
        dessevs_[-1].append(value)

In [ ]:
dessevs = {str(dessevs_[i][0]): dessevs_[i][1] for i in range(len(dessevs_)) if isinstance(dessevs_[i][0], int) and str(dessevs_[i][0]) in patient_ids}

In [ ]:
exclude_no_dessevs = []
for patient in patient_ids:
    if not patient in dessevs.keys() and not patient in exclude_no_demographics:
        exclude_no_dessevs.append(patient)
print(len(exclude_no_dessevs))

## Compiling arousal information

In [ ]:
n_patients = np.inf  # A maximum number. Actual amount may be less.
times = {}
durations = {}
stages = {}
stages2 = {}
epochs = {}
sats = {}
desats = {}
arousal_inds = {} # respiratory arousals taken into analyses
arousal_inds_resp_all = {}
arousal_inds_spont = {}
arousal_inds_all = {}
hypopnea_inds = {}
apnea_inds = {}
central_apnea_inds = {}
mixed_apnea_inds = {}
related_events = {}
pleths = {}

if n_patients < len(patient_ids):
    patients = list(np.random.choice(patient_ids, n_patients, replace=False))
else:
    patients = patient_ids.copy()
for patient in exclude_no_demographics:
    if patient in patients:
        patients.remove(patient)
for patient in exclude_no_dessevs:
    if patient in patients:
        patients.remove(patient)

for patient in tqdm(patients):
    filepath = files[patient][0]
    with open(filepath, newline='') as f:
        header = json.load(f)
    start = datetime.datetime.fromisoformat(header['startdate'])
    sample_rate_original = [signal_header for signal_header in header['SignalHeaders'] if signal_header['label'] == 'Pleth'][0]['sample_rate']
    filepath = files[patient][1]
    with open(filepath, newline='') as f:
        reader = csv.reader(f)
        events = list(reader)
    # ISO format:
    for event in events:
        if event[0][1] == ':':
            event[0] = '0'+event[0]
    filepath = files[patient][2]
    with open(filepath, newline='') as f:
        stages[patient] = [stage2num(str.strip('\r\n')) for str in f.readlines()]

    date = start.date()  # incorrect after midnight, but works for the substractions
    _times = [datetime.datetime.combine(date, datetime.time.fromisoformat(event[0])) for event in events]
    times[patient] = [(time-start).total_seconds() % (60*60*24) for time in _times]
    durations[patient] = [duration_from_str(event[4]) for event in events]
    stages2[patient] = [stage2num(event[2], ['AWAKE', 'REM', 'N1', 'N2', 'N3']) for event in events]
    epochs[patient] = [int(event[1])-1 for event in events]
    sats[patient] = [int(event[5]) if event[5] != '-' else 0 for event in events]
    desats[patient] = [int(event[6]) if event[6] != '-' else 0 for event in events]

    arousal_inds[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_resp_all[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0]]
    arousal_inds_spont[patient] = [i for i in np.where([event[3] == 'Arousal 1 ARO SPONT' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_all[patient] = [i for i in np.where([event[3][0:7] == 'Arousal' for event in events])[0]]
    hypopnea_inds[patient] = np.where([event[3] == 'Hypopnea' for event in events])[0]
    apnea_inds[patient] = np.where([event[3] == 'Obstructive Apnea' for event in events])[0]
    central_apnea_inds[patient] = np.where([event[3] == 'Central Apnea' for event in events])[0]
    mixed_apnea_inds[patient] = np.where([event[3] == 'Mixed Apnea' for event in events])[0]
    related_events[patient] = []
    for i in arousal_inds[patient]:
        e = []
        for j in hypopnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        for j in apnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        related_events[patient].append(sorted(e))
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if len(related_events[patient][i]) == 1]
    related_events[patient] = [e[0] for e in related_events[patient] if len(e) == 1]
    for i in arousal_inds[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    for i in arousal_inds_spont[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    related_events[patient] = [related_events[patient][i] for i in range(len(related_events[patient])) if stages2[patient][arousal_inds[patient][i]] < 0]
    arousal_inds[patient] = [i for i in arousal_inds[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if 0 <= sats[patient][related_events[patient][i]] <= 100]
    related_events[patient] = [e for e in related_events[patient] if 0 <= sats[patient][e] <= 100]
    arousal_inds_spont[patient] = [i for i in arousal_inds_spont[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(related_events[patient])) if durations[patient][related_events[patient][i]] > 10]
    related_events[patient] = [i for i in related_events[patient] if durations[patient][i] > 10]
    # When replicating the study, add something like the following line.
    # pleths[patient] = scipy.signal.decimate(np.load(files[patient][3]), int(sample_rate_original/sample_rate))

In [ ]:
arousals_with_5s5s_margins = {}
arousals_with_10s5s_margins = {}
arousals_with_5s10s_margins = {}
arousals_with_10s10s_margins = {}
for patient in tqdm(patients):
    arousals_with_5s5s_margins[patient] = []
    arousals_with_10s5s_margins[patient] = []
    arousals_with_5s10s_margins[patient] = []
    arousals_with_10s10s_margins[patient] = []
    for i in range(len(arousal_inds[patient])):
        start = times[patient][arousal_inds[patient][i]]
        end = start + durations[patient][arousal_inds[patient][i]]
        within55 = False
        within105 = False
        within510 = False
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds[patient][i] and j != related_events[patient][i]:
                if start - 5 < times[patient][j] < end + 5 or start - 5 < times[patient][j] + durations[patient][j] < end + 5:
                    within55 = True
                if start - 10 < times[patient][j] < end + 5 or start - 10 < times[patient][j] + durations[patient][j] < end + 5:
                    within105 = True
                if start - 5 < times[patient][j] < end + 10 or start - 5 < times[patient][j] + durations[patient][j] < end + 10:
                    within510 = True
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within55:
            arousals_with_5s5s_margins[patient].append(i)
        if not within105:
            arousals_with_10s5s_margins[patient].append(i)
        if not within510:
            arousals_with_5s10s_margins[patient].append(i)
        if not within1010:
            arousals_with_10s10s_margins[patient].append(i)
print(sum([len(arousal_inds[patient]) for patient in patients]))
print(sum([len(arousals_with_5s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_5s10s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]))

In [ ]:
arousals_with_10s10s_margins_spont = {}
for patient in tqdm(patients):
    arousals_with_10s10s_margins_spont[patient] = []
    for i in range(len(arousal_inds_spont[patient])):
        start = times[patient][arousal_inds_spont[patient][i]]
        end = start + durations[patient][arousal_inds_spont[patient][i]]
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds_spont[patient][i]:
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within1010:
            arousals_with_10s10s_margins_spont[patient].append(i)
print(sum([len(arousal_inds_spont[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients]))

In [ ]:
print(len(patients))
patients = [patient for patient in patients if len(arousals_with_10s10s_margins[patient]) > 0 or len(arousals_with_10s10s_margins_spont[patient]) > 0]
print(len(patients))

In [ ]:
# A quick and lazy way of noting that one patient originally included is missing PPG data at the revision stage when the data has been brought back through a different process.
# No reason to check everything again for a couple dozen arousals in a set of tens of thousands, just need to update the numbers. For replicating the study this doesn't matter.
print(856 - len(patients))

## Three-step fitting model

In [ ]:
# The thing with this and the next three cells is that the fitting of the model takes ridiculously long. Like, in the order of a week or multiple for about 1000 subjects, if the code is running
# continuously. So, the fittings are saved after creating them, and loaded on the subsequent runs. This even works with random interruptions of the process, you can just continue from where you got.

# Edit: if you are starting from scratch, I recommend putting some multiprocessing in this. It should help.

frequency_popts = {}
amplitude_popts = {}
for patient in tqdm(patients_fit):
    if patient in patients:
        with open(join(fit_path, patient)) as resultfile:
            results = json.load(resultfile)
        for arousal in arousals_with_10s10s_margins[patient]:
            if not arousal in [result[0] for result in results]:
                raise Exception("Missing arousals for patient " + patient)
        frequency_popts[patient] = []
        amplitude_popts[patient] = []
        for result in results:
            if result[0] in arousals_with_10s10s_margins[patient]:
                frequency_popts[patient].append(result[1])
                amplitude_popts[patient].append(result[2])

In [ ]:
# frequency_popts = {}
# amplitude_popts = {}
for patient in tqdm([patient for patient in patients if not patient in frequency_popts.keys()]):
    pleth_time = np.arange(len(pleths[patient])) / sample_rate
    arousal_pleth_inds = [np.where((times[patient][i] < pleth_time) & (pleth_time < times[patient][i] + durations[patient][i]))[0] for i in arousal_inds[patient]]
    frequency_popts[patient] = []
    amplitude_popts[patient] = []
    for arousal_i in arousals_with_10s10s_margins[patient]:
        inds = arousal_pleth_inds[arousal_i]
        inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
        if inds[-1] < len(pleths[patient]) and inds[0] > 0:
            f, t, Sxx = scipy.signal.spectrogram(pleths[patient][inds], fs=sample_rate, nperseg=sample_rate*2, nfft=sample_rate*16, noverlap=sample_rate*2-1, window='boxcar')
            t = t - before_arousal

            ydata_ = np.average(np.tile(f, (len(t), 1)).transpose(), axis=0, weights=Sxx)
            ydata = scipy.signal.savgol_filter(ydata_, 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7 # If frequency or amplitude is negative at any point, something has gone wrong in the fitting. This is extremely rare.
            frequency_popts[patient].append(popt)

            ydata_ = scipy.ndimage.maximum_filter(pleths[patient][inds], size=sample_rate*2) - scipy.ndimage.minimum_filter(pleths[patient][inds], size=sample_rate*2)
            ydata = scipy.signal.savgol_filter(ydata_[sample_rate:-sample_rate+1], 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            amplitude_popts[patient].append(popt)

        else:
            popt = np.array([np.nan]*7)
            frequency_popts[patient].append(popt)
            amplitude_popts[patient].append(popt)
    with open(fit_path + patient, 'w') as resultfile:
        json.dump([[arousals_with_10s10s_margins[patient][i],  
                    list(frequency_popts[patient][i]),
                    list(amplitude_popts[patient][i])] for i in range(len(arousals_with_10s10s_margins[patient]))], resultfile)

In [ ]:
frequency_popts_spont = {}
amplitude_popts_spont = {}
for patient in tqdm(patients_fit_spont):
    if patient in patients:
        with open(join(fit_path_spont, patient)) as resultfile:
            results = json.load(resultfile)
        for arousal in arousals_with_10s10s_margins_spont[patient]:
            if not arousal in [result[0] for result in results]:
                raise Exception("Missing arousals for patient " + patient)
        frequency_popts_spont[patient] = []
        amplitude_popts_spont[patient] = []
        for result in results:
            if result[0] in arousals_with_10s10s_margins_spont[patient]:
                frequency_popts_spont[patient].append(result[1])
                amplitude_popts_spont[patient].append(result[2])

In [ ]:
for patient in tqdm([patient for patient in patients if not patient in frequency_popts_spont.keys()]):
    pleth_time = np.arange(len(pleths[patient])) / sample_rate
    arousal_pleth_inds = [np.where((times[patient][i] < pleth_time) & (pleth_time < times[patient][i] + durations[patient][i]))[0] for i in arousal_inds_spont[patient]]
    frequency_popts_spont[patient] = []
    amplitude_popts_spont[patient] = []
    for arousal_i in arousals_with_10s10s_margins_spont[patient]:
        inds = arousal_pleth_inds[arousal_i]
        inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
        if inds[-1] < len(pleths[patient]) and inds[0] > 0:
            f, t, Sxx = scipy.signal.spectrogram(pleths[patient][inds], fs=sample_rate, nperseg=sample_rate*2, nfft=sample_rate*16, noverlap=sample_rate*2-1, window='boxcar')
            t = t - before_arousal

            ydata_ = np.average(np.tile(f, (len(t), 1)).transpose(), axis=0, weights=Sxx)
            ydata = scipy.signal.savgol_filter(ydata_, 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            frequency_popts_spont[patient].append(popt)

            ydata_ = scipy.ndimage.maximum_filter(pleths[patient][inds], size=sample_rate*2) - scipy.ndimage.minimum_filter(pleths[patient][inds], size=sample_rate*2)
            ydata = scipy.signal.savgol_filter(ydata_[sample_rate:-sample_rate+1], 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            amplitude_popts_spont[patient].append(popt)

        else:
            popt = np.array([np.nan]*7)
            frequency_popts_spont[patient].append(popt)
            amplitude_popts_spont[patient].append(popt)
    with open(fit_path_spont + patient, 'w') as resultfile:
        json.dump([[arousals_with_10s10s_margins_spont[patient][i],  
                    list(frequency_popts_spont[patient][i]),
                    list(amplitude_popts_spont[patient][i])] for i in range(len(arousals_with_10s10s_margins_spont[patient]))], resultfile)

In [ ]:
arousal_durations = []
amp_classifications = []
amp_startbaselines = []
amp_endbaselines = []
amp_expected_changes = []
amp_unexpected_changes = []
amp_second_expected_changes = []
amp_second_unexpected_changes = []
amp_meanfitvalues = []
freq_classifications = []
freq_startbaselines = []
freq_endbaselines = []
freq_expected_changes = []
freq_unexpected_changes = []
freq_second_expected_changes = []
freq_second_unexpected_changes = []
freq_meanfitvalues = []
labels = []
for patient in tqdm(arousals_with_10s10s_margins.keys()):
    for i in range(len(arousals_with_10s10s_margins[patient])):
        amp_fit = amplitude_popts[patient][i]
        freq_fit = frequency_popts[patient][i]
        duration = durations[patient][arousal_inds[patient][arousals_with_10s10s_margins[patient][i]]]
        arousal_durations.append(duration)
        amp_classification = classify(amp_fit, False, duration)
        amp_classifications.append(amp_classification)
        amp_startbaselines.append(startbaseline(amp_fit, duration, amp_classification))
        amp_endbaselines.append(endbaseline(amp_fit, duration, amp_classification))
        amp_expected_changes.append(expected_change(amp_fit, duration, amp_classification))
        amp_unexpected_changes.append(unexpected_change(amp_fit, duration, amp_classification))
        amp_second_expected_changes.append(second_expected_change(amp_fit, duration, amp_classification))
        amp_second_unexpected_changes.append(second_unexpected_change(amp_fit, duration, amp_classification))
        amp_meanfitvalues.append(meanfitvalue(amp_fit, duration))
        freq_classification = classify(freq_fit, True, duration)
        freq_classifications.append(freq_classification)
        freq_startbaselines.append(startbaseline(freq_fit, duration, freq_classification))
        freq_endbaselines.append(endbaseline(freq_fit, duration, freq_classification))
        freq_expected_changes.append(expected_change(freq_fit, duration, freq_classification))
        freq_unexpected_changes.append(unexpected_change(freq_fit, duration, freq_classification))
        freq_second_expected_changes.append(second_expected_change(freq_fit, duration, freq_classification))
        freq_second_unexpected_changes.append(second_unexpected_change(freq_fit, duration, freq_classification))
        freq_meanfitvalues.append(meanfitvalue(freq_fit, duration))
        labels.append((patient, i))

In [ ]:
amp_eventual_expected_changes = []
amp_second_chosen = []
freq_eventual_expected_changes = []
freq_second_chosen = []
for i in tqdm(range(len(amp_expected_changes))):
    if (not np.isnan(amp_second_expected_changes[i][0])) and abs(amp_second_expected_changes[i][0] - amp_startbaselines[i])*(arousal_durations[i]+before_arousal-amp_second_expected_changes[i][1]) > abs(amp_expected_changes[i][0] - amp_startbaselines[i])*(amp_expected_changes[i][2]-amp_expected_changes[i][1]):
        amp_eventual_expected_changes.append(amp_second_expected_changes[i])
        amp_second_chosen.append(True)
    else:
        amp_eventual_expected_changes.append(amp_expected_changes[i])
        amp_second_chosen.append(False)
    if (not np.isnan(freq_second_expected_changes[i][0])) and abs(freq_second_expected_changes[i][0] - freq_startbaselines[i])*(arousal_durations[i]+before_arousal-freq_second_expected_changes[i][1]) > abs(freq_expected_changes[i][0] - freq_startbaselines[i])*(freq_expected_changes[i][2]-freq_expected_changes[i][1]):
        freq_eventual_expected_changes.append(freq_second_expected_changes[i])
        freq_second_chosen.append(True)
    else:
        freq_eventual_expected_changes.append(freq_expected_changes[i])
        freq_second_chosen.append(False)
print(np.count_nonzero(amp_second_chosen))
print(np.count_nonzero(freq_second_chosen))

In [ ]:
arousal_durations_spont = []
amp_classifications_spont = []
amp_startbaselines_spont = []
amp_endbaselines_spont = []
amp_expected_changes_spont = []
amp_unexpected_changes_spont = []
amp_second_expected_changes_spont = []
amp_second_unexpected_changes_spont = []
amp_meanfitvalues_spont = []
freq_classifications_spont = []
freq_startbaselines_spont = []
freq_endbaselines_spont = []
freq_expected_changes_spont = []
freq_unexpected_changes_spont = []
freq_second_expected_changes_spont = []
freq_second_unexpected_changes_spont = []
freq_meanfitvalues_spont = []
labels_spont = []
for patient in tqdm(arousals_with_10s10s_margins_spont.keys()):
    for i in range(len(arousals_with_10s10s_margins_spont[patient])):
        amp_fit = amplitude_popts_spont[patient][i]
        freq_fit = frequency_popts_spont[patient][i]
        duration = durations[patient][arousal_inds_spont[patient][arousals_with_10s10s_margins_spont[patient][i]]]
        arousal_durations_spont.append(duration)
        amp_classification = classify(amp_fit, False, duration)
        amp_classifications_spont.append(amp_classification)
        amp_startbaselines_spont.append(startbaseline(amp_fit, duration, amp_classification))
        amp_endbaselines_spont.append(endbaseline(amp_fit, duration, amp_classification))
        amp_expected_changes_spont.append(expected_change(amp_fit, duration, amp_classification))
        amp_unexpected_changes_spont.append(unexpected_change(amp_fit, duration, amp_classification))
        amp_second_expected_changes_spont.append(second_expected_change(amp_fit, duration, amp_classification))
        amp_second_unexpected_changes_spont.append(second_unexpected_change(amp_fit, duration, amp_classification))
        amp_meanfitvalues_spont.append(meanfitvalue(amp_fit, duration))
        freq_classification = classify(freq_fit, True, duration)
        freq_classifications_spont.append(freq_classification)
        freq_startbaselines_spont.append(startbaseline(freq_fit, duration, freq_classification))
        freq_endbaselines_spont.append(endbaseline(freq_fit, duration, freq_classification))
        freq_expected_changes_spont.append(expected_change(freq_fit, duration, freq_classification))
        freq_unexpected_changes_spont.append(unexpected_change(freq_fit, duration, freq_classification))
        freq_second_expected_changes_spont.append(second_expected_change(freq_fit, duration, freq_classification))
        freq_second_unexpected_changes_spont.append(second_unexpected_change(freq_fit, duration, freq_classification))
        freq_meanfitvalues_spont.append(meanfitvalue(freq_fit, duration))
        labels_spont.append((patient, i))

In [ ]:
amp_eventual_expected_changes_spont = []
amp_second_chosen_spont = []
freq_eventual_expected_changes_spont = []
freq_second_chosen_spont = []
for i in tqdm(range(len(amp_expected_changes_spont))):
    if (not np.isnan(amp_second_expected_changes_spont[i][0])) and abs(amp_second_expected_changes_spont[i][0] - amp_startbaselines_spont[i])*(arousal_durations_spont[i]+before_arousal-amp_second_expected_changes_spont[i][1]) > abs(amp_expected_changes_spont[i][0] - amp_startbaselines_spont[i])*(amp_expected_changes_spont[i][2]-amp_expected_changes_spont[i][1]):
        amp_eventual_expected_changes_spont.append(amp_second_expected_changes_spont[i])
        amp_second_chosen_spont.append(True)
    else:
        amp_eventual_expected_changes_spont.append(amp_expected_changes_spont[i])
        amp_second_chosen_spont.append(False)
    if (not np.isnan(freq_second_expected_changes_spont[i][0])) and abs(freq_second_expected_changes_spont[i][0] - freq_startbaselines_spont[i])*(arousal_durations_spont[i]+before_arousal-freq_second_expected_changes_spont[i][1]) > abs(freq_expected_changes_spont[i][0] - freq_startbaselines_spont[i])*(freq_expected_changes_spont[i][2]-freq_expected_changes_spont[i][1]):
        freq_eventual_expected_changes_spont.append(freq_second_expected_changes_spont[i])
        freq_second_chosen_spont.append(True)
    else:
        freq_eventual_expected_changes_spont.append(freq_expected_changes_spont[i])
        freq_second_chosen_spont.append(False)
print(np.count_nonzero(amp_second_chosen_spont))
print(np.count_nonzero(freq_second_chosen_spont))

In [ ]:
print('\nRESPIRATORY AROUSALS')
print('\nAmplitude:')
print(repr(len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) / len(amp_eventual_expected_changes) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print(repr(len([change for change in amp_eventual_expected_changes if change[4] == 1]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by an increase.')
print(repr(len([i for i in range(len(amp_eventual_expected_changes)) if amp_eventual_expected_changes[i][4] == -1 or amp_second_unexpected_changes[i][4] == 1]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by an increase.')
print(repr(len([i for i in range(len(amp_expected_changes)) if not np.isnan(amp_expected_changes[i][0]) and amp_second_chosen[i]]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by another, "smaller" decrease.')
print(repr(len([i for i in range(len(amp_expected_changes)) if not np.isnan(amp_expected_changes[i][0]) and not np.isnan(amp_second_expected_changes[i][0]) and not amp_second_chosen[i]]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by another, "smaller" decrease.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in amp_eventual_expected_changes if not np.isnan(change[3])]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(amp_expected_changes)) if np.isnan(amp_expected_changes[i][0]) and not np.isnan(amp_unexpected_changes[i][0])]) / len(amp_expected_changes) * 100) + ' % of the arousals showed an unaccompanied positive change (or two).')
print('\nFrequency:')
print(repr(len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) / len(freq_eventual_expected_changes) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print(repr(len([change for change in freq_eventual_expected_changes if change[4] == 1]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by a decrease.')
print(repr(len([i for i in range(len(freq_eventual_expected_changes)) if freq_eventual_expected_changes[i][4] == -1 or freq_second_unexpected_changes[i][4] == 1]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by a decrease.')
print(repr(len([i for i in range(len(freq_expected_changes)) if not np.isnan(freq_expected_changes[i][0]) and freq_second_chosen[i]]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by another, "smaller" increase.')
print(repr(len([i for i in range(len(freq_expected_changes)) if not np.isnan(freq_expected_changes[i][0]) and not np.isnan(freq_second_expected_changes[i][0]) and not freq_second_chosen[i]]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by another, "smaller" increase.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in freq_eventual_expected_changes if not np.isnan(change[3])]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(freq_expected_changes)) if np.isnan(freq_expected_changes[i][0]) and not np.isnan(freq_unexpected_changes[i][0])]) / len(freq_expected_changes) * 100) + ' % of the arousals showed an unaccompanied negative change (or two).')

print('\n' + repr(len([i for i in range(len(amp_eventual_expected_changes)) if not np.isnan(amp_eventual_expected_changes[i][0]) or not np.isnan(freq_eventual_expected_changes[i][0])]) / len(amp_eventual_expected_changes) * 100) + ' % of the arousals showed at least one type of expected change.')

In [ ]:
print('\nSPONTANEOUS AROUSALS')
print('\nAmplitude:')
print(repr(len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) / len(amp_eventual_expected_changes_spont) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print(repr(len([change for change in amp_eventual_expected_changes_spont if change[4] == 1]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by an increase.')
print(repr(len([i for i in range(len(amp_eventual_expected_changes_spont)) if amp_eventual_expected_changes_spont[i][4] == -1 or amp_second_unexpected_changes_spont[i][4] == 1]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by an increase.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if not np.isnan(amp_expected_changes_spont[i][0]) and amp_second_chosen_spont[i]]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by another, "smaller" decrease.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if not np.isnan(amp_expected_changes_spont[i][0]) and not np.isnan(amp_second_expected_changes_spont[i][0]) and not amp_second_chosen_spont[i]]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by another, "smaller" decrease.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[3])]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if np.isnan(amp_expected_changes_spont[i][0]) and not np.isnan(amp_unexpected_changes_spont[i][0])]) / len(amp_expected_changes_spont) * 100) + ' % of the arousals showed an unaccompanied positive change (or two).')
print('\nFrequency:')
print(repr(len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) / len(freq_eventual_expected_changes_spont) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print(repr(len([change for change in freq_eventual_expected_changes_spont if change[4] == 1]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by a decrease.')
print(repr(len([i for i in range(len(freq_eventual_expected_changes_spont)) if freq_eventual_expected_changes_spont[i][4] == -1 or freq_second_unexpected_changes_spont[i][4] == 1]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by a decrease.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if not np.isnan(freq_expected_changes_spont[i][0]) and freq_second_chosen_spont[i]]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by another, "smaller" increase.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if not np.isnan(freq_expected_changes_spont[i][0]) and not np.isnan(freq_second_expected_changes_spont[i][0]) and not freq_second_chosen_spont[i]]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by another, "smaller" increase.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[3])]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if np.isnan(freq_expected_changes_spont[i][0]) and not np.isnan(freq_unexpected_changes_spont[i][0])]) / len(freq_expected_changes_spont) * 100) + ' % of the arousals showed an unaccompanied negative change (or two).')

print('\n' + repr(len([i for i in range(len(amp_eventual_expected_changes_spont)) if not np.isnan(amp_eventual_expected_changes_spont[i][0]) or not np.isnan(freq_eventual_expected_changes_spont[i][0])]) / len(amp_eventual_expected_changes_spont) * 100) + ' % of the arousals showed at least one type of expected change.')

In [ ]:
print('\ALL AROUSALS')
print('\nAmplitude:')
print(repr((len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])])) / (len(amp_eventual_expected_changes) + len(amp_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print('In ' + repr((len([change for change in amp_eventual_expected_changes if not np.isnan(change[3])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[3])])) / (len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])])) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print('\nFrequency:')
print(repr((len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])])) / (len(freq_eventual_expected_changes) + len(freq_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print('In ' + repr((len([change for change in freq_eventual_expected_changes if not np.isnan(change[3])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[3])])) / (len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])])) * 100) + ' % of these increases the signal was observed to return towards the baseline.')

print('\n' + repr((len([i for i in range(len(amp_eventual_expected_changes)) if not np.isnan(amp_eventual_expected_changes[i][0]) or not np.isnan(freq_eventual_expected_changes[i][0])]) + len([i for i in range(len(amp_eventual_expected_changes_spont)) if not np.isnan(amp_eventual_expected_changes_spont[i][0]) or not np.isnan(freq_eventual_expected_changes_spont[i][0])])) / (len(amp_eventual_expected_changes) + len(amp_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed at least one type of expected change.')

## Demographics etc.

In [ ]:
print('Male:')
m = len([patient for patient in patients if demographics_by_patient[patient][1] == 1])
print(m)
print(repr(m/len(patients)) + ' %')
f = len([patient for patient in patients if demographics_by_patient[patient][1] == 0])
print(f)
print(repr(f/len(patients)) + ' %')

In [ ]:
print('Age:')
print(np.percentile([demographics_by_patient[patient][2] for patient in patients], [25, 50, 75]))
print('BMI:')
print(np.percentile([demographics_by_patient[patient][3] for patient in patients], [25, 50, 75]))

In [ ]:
tst = []
waso = []
ahi = []
ari = []
arri = []
number_of_lapses_by_patient = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients]
mean_speed_by_patient = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients]
stage_r = []
nrem = []
stage_n1 = []
stage_n2 = []
stage_n3 = []

for patient in patients:
    tst.append(len([stage for stage in stages[patient] if stage != 0])*30/3600)
    waso.append((len([stage for stage in stages[patient] if stage == 0])-np.nonzero(stages[patient])[0][0])*30/3600)
    ahi.append((len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/tst[-1])
    ari.append(len(arousal_inds_all[patient])/tst[-1])
    arri.append(len(arousal_inds_resp_all[patient])/tst[-1])
    stage_r.append(len([stage for stage in stages[patient] if stage == -1])/len([stage for stage in stages[patient] if stage != 0])*100)
    nrem.append(len([stage for stage in stages[patient] if stage < -1])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n1.append(len([stage for stage in stages[patient] if stage == -2])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n2.append(len([stage for stage in stages[patient] if stage == -3])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n3.append(len([stage for stage in stages[patient] if stage == -4])/len([stage for stage in stages[patient] if stage != 0])*100)

print('TST:')
print(np.percentile(tst, [25, 50, 75]))
print('WASO:')
print(np.percentile(waso, [25, 50, 75]))
print('AHI:')
print(np.percentile(ahi, [25, 50, 75]))
print('ArI:')
print(np.percentile(ari, [25, 50, 75]))
print('ArRI:')
print(np.percentile(arri, [25, 50, 75]))
print('DesSev:')
print(np.percentile(list(dessevs.values()), [25, 50, 75]))
print('Lapses_PVT:')
print(np.percentile(number_of_lapses_by_patient, [25, 50, 75]))
print('Mean 1/RT:')
print(np.percentile(mean_speed_by_patient, [25, 50, 75]))
print('Stage R:')
print(np.percentile(stage_r, [25, 50, 75]))
print('NREM:')
print(np.percentile(nrem, [25, 50, 75]))
print('Stage N1:')
print(np.percentile(stage_n1, [25, 50, 75]))
print('Stage N2:')
print(np.percentile(stage_n2, [25, 50, 75]))
print('Stage N3:')
print(np.percentile(stage_n3, [25, 50, 75]))

## Effect of sleep stage

In [ ]:
# indices corresponding to lists produced by the models, in four groups: R, N1, N2, N3
indices_by_stage = [[i for i in range(len(labels)) if stages2[labels[i][0]][arousal_inds[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] == stage] for stage in [-1, -2, -3, -4]]
indices_by_stage_spont = [[i for i in range(len(labels_spont)) if stages2[labels_spont[i][0]][arousal_inds_spont[labels_spont[i][0]][arousals_with_10s10s_margins_spont[labels_spont[i][0]][labels_spont[i][1]]]] == stage] for stage in [-1, -2, -3, -4]]

In [ ]:
print('Stage R:')
print(len(indices_by_stage[0] + indices_by_stage_spont[0]))
print((len(indices_by_stage[0] + indices_by_stage_spont[0]))/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('NREM:')
print(len(indices_by_stage[1] + indices_by_stage_spont[1] + indices_by_stage[2] + indices_by_stage_spont[2] + indices_by_stage[3] + indices_by_stage_spont[3]))
print(len(indices_by_stage[1] + indices_by_stage_spont[1] + indices_by_stage[2] + indices_by_stage_spont[2] + indices_by_stage[3] + indices_by_stage_spont[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N1:')
print(len(indices_by_stage[1] + indices_by_stage_spont[1]))
print(len(indices_by_stage[1] + indices_by_stage_spont[1])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N2:')
print(len(indices_by_stage[2] + indices_by_stage_spont[2]))
print(len(indices_by_stage[2] + indices_by_stage_spont[2])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N3:')
print(len(indices_by_stage[3] + indices_by_stage_spont[3]))
print(len(indices_by_stage[3] + indices_by_stage_spont[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))

In [ ]:
freq_magnitudes = [freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 if not np.isnan(freq_startbaselines[j]) else np.nan for j in range(len(labels))]
freq_magnitudes_spont = [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 if not np.isnan(freq_startbaselines_spont[j]) else np.nan for j in range(len(labels_spont))]
amp_magnitudes = [-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 if not np.isnan(amp_startbaselines[j]) else np.nan for j in range(len(labels))]
amp_magnitudes_spont = [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 if not np.isnan(amp_startbaselines_spont[j]) else np.nan for j in range(len(labels_spont))]

In [ ]:
freq_magnitudes_by_stage = [[freq_magnitudes[j] for j in indices_by_stage[i] if not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_stage_spont[i] if not np.isnan(freq_magnitudes_spont[j])] for i in range(4)]
amp_magnitudes_by_stage = [[amp_magnitudes[j] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont[j])] for i in range(4)]

In [ ]:
freq_magnitudes_by_stage_patients = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(freq_magnitudes[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(freq_magnitudes_spont[j])] for i in range(4)]
amp_magnitudes_by_stage_patients = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont[j])] for i in range(4)]

In [ ]:
# This takes a long time. At the end it saves the results, so you shouldn't have to run this many times.
# UPDATE: now it uses multithreading, so it should be much faster. You may need to increase your ulimit -n (Assuming Linux).

freq_dir_mat = np.zeros([4,4])
freq_wilcoxonstat_mat = []
freq_pval_mat = []
freq_pval_precise_mat = []
freq_median_difference_mat = []
freq_median_difference_lo_mat = []
freq_median_difference_hi_mat = []
freq_cohen_d_mat = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    freq_median_difference_mat.append(((i, j), np.median(freq_magnitudes_by_stage[i]) - np.median(freq_magnitudes_by_stage[j])))
    lo, hi = median_difference_CI(freq_magnitudes_by_stage[i], freq_magnitudes_by_stage[j], 0.95)
    freq_median_difference_lo_mat.append(((i, j), lo))
    freq_median_difference_hi_mat.append(((i, j), hi))
    if np.median(freq_magnitudes_by_stage[i]) < np.median(freq_magnitudes_by_stage[j]):
        # alternative = 'less'
        freq_dir_mat[i, j] = -1
    else:
        # alternative = 'greater'
        freq_dir_mat[i, j] = 1
    # One-sided alternatives with the direction chosen based on the data to be tested was a questionable choice, changed in the revision for the sake of rigor.
    alternative = 'two-sided'
    pvalue, _, T, _, z, _ = iterative_wilcoxon(freq_magnitudes_by_stage[i], freq_magnitudes_by_stage[j], freq_magnitudes_by_stage_patients[i], freq_magnitudes_by_stage_patients[j], alternative, 1000)
    freq_pval_precise_mat.append(((i, j), pvalue))
    if pvalue < 0.001:
        freq_pval_mat.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        freq_pval_mat.append(((i, j), '%.3f' % pvalue))
    else:
        freq_pval_mat.append(((i, j), '%.2f' % pvalue))
    freq_wilcoxonstat_mat.append(((i, j), T))
    _, d = effect_size_r_d(z, len(freq_magnitudes_by_stage[i]), len(freq_magnitudes_by_stage[j]))
    freq_cohen_d_mat.append(((i, j), d))

with open('/wrk/luukinen/results/20260621/freqstagemats.pickle', 'wb') as f:
    pickle.dump((freq_dir_mat, freq_wilcoxonstat_mat, freq_pval_mat, freq_pval_precise_mat, freq_median_difference_mat, freq_median_difference_lo_mat, freq_median_difference_hi_mat, freq_cohen_d_mat), f)

In [ ]:
with open('/wrk/luukinen/results/20260621/freqstagemats.pickle', 'rb') as f:
    freq_dir_mat, freq_wilcoxonstat_mat, freq_pval_mat, freq_pval_precise_mat, freq_median_difference_mat, freq_median_difference_lo_mat, freq_median_difference_hi_mat, freq_cohen_d_mat = pickle.load(f)

In [ ]:
for mat in [freq_dir_mat, freq_wilcoxonstat_mat, freq_pval_mat, freq_pval_precise_mat, freq_median_difference_mat, freq_median_difference_lo_mat, freq_median_difference_hi_mat, freq_cohen_d_mat]:
    print(mat)
    print('')

In [ ]:
# Same thing as above applies here too.

amp_dir_mat = np.zeros([4,4])
amp_wilcoxonstat_mat = []
amp_pval_mat = []
amp_pval_precise_mat = []
amp_median_difference_mat = []
amp_median_difference_lo_mat = []
amp_median_difference_hi_mat = []
amp_cohen_d_mat = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    amp_median_difference_mat.append(((i, j), np.median(amp_magnitudes_by_stage[i]) - np.median(amp_magnitudes_by_stage[j])))
    lo, hi = median_difference_CI(amp_magnitudes_by_stage[i], amp_magnitudes_by_stage[j], 0.95)
    amp_median_difference_lo_mat.append(((i, j), lo))
    amp_median_difference_hi_mat.append(((i, j), hi))
    if np.median(amp_magnitudes_by_stage[i]) < np.median(amp_magnitudes_by_stage[j]):
        # alternative = 'less'
        amp_dir_mat[i, j] = -1
    else:
        # alternative = 'greater'
        amp_dir_mat[i, j] = 1
    # One-sided alternatives with the direction chosen based on the data to be tested was a questionable choice, changed in the revision for the sake of rigor.
    alternative = 'two-sided'
    pvalue, _, T, _, z, _ = iterative_wilcoxon(amp_magnitudes_by_stage[i], amp_magnitudes_by_stage[j], amp_magnitudes_by_stage_patients[i], amp_magnitudes_by_stage_patients[j], alternative, 1000)
    amp_pval_precise_mat.append(((i, j), pvalue))
    if pvalue < 0.001:
        amp_pval_mat.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        amp_pval_mat.append(((i, j), '%.3f' % pvalue))
    else:
        amp_pval_mat.append(((i, j), '%.2f' % pvalue))
    amp_wilcoxonstat_mat.append(((i, j), T))
    _, d = effect_size_r_d(z, len(amp_magnitudes_by_stage[i]), len(amp_magnitudes_by_stage[j]))
    amp_cohen_d_mat.append(((i, j), d))

with open('/wrk/luukinen/results/20260621/ampstagemats.pickle', 'wb') as f:
    pickle.dump((amp_dir_mat, amp_wilcoxonstat_mat, amp_pval_mat, amp_pval_precise_mat, amp_median_difference_mat, amp_median_difference_lo_mat, amp_median_difference_hi_mat, amp_cohen_d_mat), f)

In [ ]:
with open('/wrk/luukinen/results/20260621/ampstagemats.pickle', 'rb') as f:
    amp_dir_mat, amp_wilcoxonstat_mat, amp_pval_mat, amp_pval_precise_mat, amp_median_difference_mat, amp_median_difference_lo_mat, amp_median_difference_hi_mat, amp_cohen_d_mat = pickle.load(f)

In [ ]:
for mat in [amp_dir_mat, amp_wilcoxonstat_mat, amp_pval_mat, amp_pval_precise_mat, amp_median_difference_mat, amp_median_difference_lo_mat, amp_median_difference_hi_mat, amp_cohen_d_mat]:
    print(mat)
    print('')

In [ ]:
[list(np.unique(l)) for l in amp_magnitudes_by_stage_patients] == [list(np.unique(l)) for l in freq_magnitudes_by_stage_patients]

In [ ]:
patients_with_arousals_in_all_stages = [patient for patient in patients if np.all([patient in l for l in freq_magnitudes_by_stage_patients])]
patients_with_arousals_in_all_NREM_stages = [patient for patient in patients if patient in freq_magnitudes_by_stage_patients[1] and patient in freq_magnitudes_by_stage_patients[2] and patient in freq_magnitudes_by_stage_patients[3]]
patients_with_arousals_in_N1_and_N2 = [patient for patient in patients if patient in freq_magnitudes_by_stage_patients[1] and patient in freq_magnitudes_by_stage_patients[2]]

### Relativization

In [ ]:
pleth_range_90 = {}
def inter_percentile_range(data, low, high):
    return np.percentile(data, high) - np.percentile(data, low)
for patient in tqdm(patients):
    pleth_range_90[patient] = inter_percentile_range(np.load(join(pleth_path, patient, plethend)), 5, 95)
amp_magnitudes_relative = [amp_magnitudes[i] / pleth_range_90[labels[i][0]] for i in range(len(amp_magnitudes))]
amp_magnitudes_spont_relative = [amp_magnitudes_spont[i] / pleth_range_90[labels_spont[i][0]] for i in range(len(amp_magnitudes_spont))]

In [ ]:
amp_magnitudes_by_stage_relative = [[amp_magnitudes_relative[j] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont_relative[j])] for i in range(4)]

In [ ]:
amp_magnitudes_by_stage_patients_relative = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes_relative[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont_relative[j])] for i in range(4)]

In [ ]:
# Again a relatively slow one, especially if multithreading is not possible. Saves the results.

amp_dir_mat_relative = np.zeros([4,4])
amp_wilcoxonstat_mat_relative = []
amp_pval_mat_relative = []
amp_pval_precise_mat_relative = []
amp_median_difference_mat_relative = []
amp_median_difference_lo_mat_relative = []
amp_median_difference_hi_mat_relative = []
amp_cohen_d_mat_relative = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    amp_median_difference_mat_relative.append(((i, j), np.median(amp_magnitudes_by_stage_relative[i]) - np.median(amp_magnitudes_by_stage_relative[j])))
    lo, hi = median_difference_CI(amp_magnitudes_by_stage_relative[i], amp_magnitudes_by_stage_relative[j], 0.95)
    amp_median_difference_lo_mat_relative.append(((i, j), lo))
    amp_median_difference_hi_mat_relative.append(((i, j), hi))
    if np.median(amp_magnitudes_by_stage_relative[i]) < np.median(amp_magnitudes_by_stage_relative[j]):
        # alternative = 'less'
        amp_dir_mat_relative[i, j] = -1
    else:
        # alternative = 'greater'
        amp_dir_mat_relative[i, j] = 1
    # One-sided alternatives with the direction chosen based on the data to be tested was a questionable choice, changed in the revision for the sake of rigor.
    alternative = 'two-sided'
    pvalue, _, T, _, z, _ = iterative_wilcoxon(amp_magnitudes_by_stage_relative[i], amp_magnitudes_by_stage_relative[j], amp_magnitudes_by_stage_patients_relative[i], amp_magnitudes_by_stage_patients_relative[j], alternative, 1000)
    amp_pval_precise_mat_relative.append(((i, j), pvalue))
    if pvalue < 0.001:
        amp_pval_mat_relative.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        amp_pval_mat_relative.append(((i, j), '%.3f' % pvalue))
    else:
        amp_pval_mat_relative.append(((i, j), '%.2f' % pvalue))
    amp_wilcoxonstat_mat_relative.append(((i, j), T))
    _, d = effect_size_r_d(z, len(amp_magnitudes_by_stage_relative[i]), len(amp_magnitudes_by_stage_relative[j]))
    amp_cohen_d_mat_relative.append(((i, j), d))

with open('/wrk/luukinen/results/20260621/ampstagemats_relative.pickle', 'wb') as f:
    pickle.dump((amp_dir_mat_relative, amp_wilcoxonstat_mat_relative, amp_pval_mat_relative, amp_pval_precise_mat_relative, amp_median_difference_mat_relative, amp_median_difference_lo_mat_relative, amp_median_difference_hi_mat_relative, amp_cohen_d_mat_relative), f)

In [ ]:
with open('/wrk/luukinen/results/20260621/ampstagemats_relative.pickle', 'rb') as f:
    amp_dir_mat_relative, amp_wilcoxonstat_mat_relative, amp_pval_mat_relative, amp_pval_precise_mat_relative, amp_median_difference_mat_relative, amp_median_difference_lo_mat_relative, amp_median_difference_hi_mat_relative, amp_cohen_d_mat_relative = pickle.load(f)

In [ ]:
for mat in [amp_dir_mat_relative, amp_wilcoxonstat_mat_relative, amp_pval_mat_relative, amp_pval_precise_mat_relative, amp_median_difference_mat_relative, amp_median_difference_lo_mat_relative, amp_median_difference_hi_mat_relative, amp_cohen_d_mat_relative]:
    print(mat)
    print('')

### Normalization

In [ ]:
# indices corresponding to lists produced by the models, in three groups: apneas with desat, apneas without desat, hypopneas with desat and hypopneas without desat. The desaturation threshold is 3 % (inclusive)
arousals_by_type = [
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3],
]
[len(l) for l in arousals_by_type]

In [ ]:
print('Apnea, Des:')
print(len(arousals_by_type[0])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Apnea, no Des:')
print(len(arousals_by_type[1])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Hypopnea, Des:')
print(len(arousals_by_type[2])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Hypopnea, no Des:')
print(len(arousals_by_type[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Spontaneous:')
print(sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))

In [ ]:
freq_changes_by_arousal_type = [[freq_magnitudes[i] for i in l if not np.isnan(freq_magnitudes[i])] for l in arousals_by_type] + [[freq_magnitudes_spont[i] for i in range(len(freq_magnitudes_spont)) if not np.isnan(freq_magnitudes_spont[i])]]
amp_changes_by_arousal_type_relative = [[amp_magnitudes_relative[i] for i in l if not np.isnan(amp_magnitudes_relative[i])] for l in arousals_by_type] + [[amp_magnitudes_spont_relative[i] for i in range(len(amp_magnitudes_spont_relative)) if not np.isnan(amp_magnitudes_spont_relative[i])]]

In [ ]:
median_freq_magnitude_by_type = [np.median(l) for l in freq_changes_by_arousal_type]
median_amp_magnitude_by_type_relative = [np.median(l) for l in amp_changes_by_arousal_type_relative]

In [ ]:
arousal_types = []
for i in range(len(labels)):
    if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3:
        arousal_types.append(0)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3:
        arousal_types.append(1)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3:
        arousal_types.append(2)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3:
        arousal_types.append(3)
    else:
        raise Exception('Related event uncategorized! This should not happen. You have probably broken something.')

In [ ]:
freq_magnitudes_normalized = [freq_magnitudes[i] / median_freq_magnitude_by_type[arousal_types[i]] for i in range(len(freq_magnitudes))]
freq_magnitudes_spont_normalized = [freq_magnitudes_spont[i] / median_freq_magnitude_by_type[4] for i in range(len(freq_magnitudes_spont))]
amp_magnitudes_normalized = [amp_magnitudes_relative[i] / median_amp_magnitude_by_type_relative[arousal_types[i]] for i in range(len(amp_magnitudes))]
amp_magnitudes_spont_normalized = [amp_magnitudes_spont_relative[i] / median_amp_magnitude_by_type_relative[4] for i in range(len(amp_magnitudes_spont))]

In [ ]:
freq_magnitudes_by_stage_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_stage[i] if not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_stage_spont[i] if not np.isnan(freq_magnitudes_spont_normalized[j])] for i in range(4)]
amp_magnitudes_by_stage_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont_normalized[j])] for i in range(4)]

In [ ]:
freq_magnitudes_by_stage_patients_normalized = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(freq_magnitudes_normalized[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(freq_magnitudes_spont_normalized[j])] for i in range(4)]
amp_magnitudes_by_stage_patients_normalized = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(amp_magnitudes_normalized[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(amp_magnitudes_spont_normalized[j])] for i in range(4)]

In [ ]:
# Again, very slow if you have only one CPU but saves its results.

freq_dir_mat_normalized = np.zeros([4,4])
freq_wilcoxonstat_mat_normalized = []
freq_pval_mat_normalized = []
freq_pval_precise_mat_normalized = []
freq_median_difference_mat_normalized = []
freq_median_difference_lo_mat_normalized = []
freq_median_difference_hi_mat_normalized = []
freq_cohen_d_mat_normalized = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    freq_median_difference_mat_normalized.append(((i, j), np.median(freq_magnitudes_by_stage_normalized[i]) - np.median(freq_magnitudes_by_stage_normalized[j])))
    lo, hi = median_difference_CI(freq_magnitudes_by_stage_normalized[i], freq_magnitudes_by_stage_normalized[j], 0.95)
    freq_median_difference_lo_mat_normalized.append(((i, j), lo))
    freq_median_difference_hi_mat_normalized.append(((i, j), hi))
    if np.median(freq_magnitudes_by_stage_normalized[i]) < np.median(freq_magnitudes_by_stage_normalized[j]):
        # alternative = 'less'
        freq_dir_mat_normalized[i, j] = -1
    else:
        # alternative = 'greater'
        freq_dir_mat_normalized[i, j] = 1
    # One-sided alternatives with the direction chosen based on the data to be tested was a questionable choice, changed in the revision for the sake of rigor.
    alternative = 'two-sided'
    pvalue, _, T, _, z, _ = iterative_wilcoxon(freq_magnitudes_by_stage_normalized[i], freq_magnitudes_by_stage_normalized[j], freq_magnitudes_by_stage_patients_normalized[i], freq_magnitudes_by_stage_patients_normalized[j], alternative, 1000)
    freq_pval_precise_mat_normalized.append(((i, j), pvalue))
    if pvalue < 0.001:
        freq_pval_mat_normalized.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        freq_pval_mat_normalized.append(((i, j), '%.3f' % pvalue))
    else:
        freq_pval_mat_normalized.append(((i, j), '%.2f' % pvalue))
    freq_wilcoxonstat_mat_normalized.append(((i, j), T))
    _, d = effect_size_r_d(z, len(freq_magnitudes_by_stage_normalized[i]), len(freq_magnitudes_by_stage_normalized[j]))
    freq_cohen_d_mat_normalized.append(((i, j), d))

with open('/wrk/luukinen/results/20260621/freqstagemats_normalized.pickle', 'wb') as f:
    pickle.dump((freq_dir_mat_normalized, freq_wilcoxonstat_mat_normalized, freq_pval_mat_normalized, freq_pval_precise_mat_normalized, freq_median_difference_mat_normalized, freq_median_difference_lo_mat_normalized, freq_median_difference_hi_mat_normalized, freq_cohen_d_mat_normalized), f)

In [ ]:
with open('/wrk/luukinen/results/20260621/freqstagemats_normalized.pickle', 'rb') as f:
    freq_dir_mat_normalized, freq_wilcoxonstat_mat_normalized, freq_pval_mat_normalized, freq_pval_precise_mat_normalized, freq_median_difference_mat_normalized, freq_median_difference_lo_mat_normalized, freq_median_difference_hi_mat_normalized, freq_cohen_d_mat_normalized = pickle.load(f)

In [ ]:
for mat in [freq_dir_mat_normalized, freq_wilcoxonstat_mat_normalized, freq_pval_mat_normalized, freq_pval_precise_mat_normalized, freq_median_difference_mat_normalized, freq_median_difference_lo_mat_normalized, freq_median_difference_hi_mat_normalized, freq_cohen_d_mat_normalized]:
    print(mat)
    print('')

In [ ]:
# Ditto.

amp_dir_mat_normalized = np.zeros([4,4])
amp_wilcoxonstat_mat_normalized = []
amp_pval_mat_normalized = []
amp_pval_precise_mat_normalized = []
amp_median_difference_mat_normalized = []
amp_median_difference_lo_mat_normalized = []
amp_median_difference_hi_mat_normalized = []
amp_cohen_d_mat_normalized = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    amp_median_difference_mat_normalized.append(((i, j), np.median(amp_magnitudes_by_stage_normalized[i]) - np.median(amp_magnitudes_by_stage_normalized[j])))
    lo, hi = median_difference_CI(amp_magnitudes_by_stage_normalized[i], amp_magnitudes_by_stage_normalized[j], 0.95)
    amp_median_difference_lo_mat_normalized.append(((i, j), lo))
    amp_median_difference_hi_mat_normalized.append(((i, j), hi))
    if np.median(amp_magnitudes_by_stage_normalized[i]) < np.median(amp_magnitudes_by_stage_normalized[j]):
        # alternative = 'less'
        amp_dir_mat_normalized[i, j] = -1
    else:
        # alternative = 'greater'
        amp_dir_mat_normalized[i, j] = 1
    # One-sided alternatives with the direction chosen based on the data to be tested was a questionable choice, changed in the revision for the sake of rigor.
    alternative = 'two-sided'
    pvalue, _, T, _, z, _ = iterative_wilcoxon(amp_magnitudes_by_stage_normalized[i], amp_magnitudes_by_stage_normalized[j], amp_magnitudes_by_stage_patients_normalized[i], amp_magnitudes_by_stage_patients_normalized[j], alternative, 1000)
    amp_pval_precise_mat_normalized.append(((i, j), pvalue))
    if pvalue < 0.001:
        amp_pval_mat_normalized.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        amp_pval_mat_normalized.append(((i, j), '%.3f' % pvalue))
    else:
        amp_pval_mat_normalized.append(((i, j), '%.2f' % pvalue))
    amp_wilcoxonstat_mat_normalized.append(((i, j), T))
    _, d = effect_size_r_d(z, len(amp_magnitudes_by_stage_normalized[i]), len(amp_magnitudes_by_stage_normalized[j]))
    amp_cohen_d_mat_normalized.append(((i, j), d))

with open('/wrk/luukinen/results/20260621/ampstagemats_normalized.pickle', 'wb') as f:
    pickle.dump((amp_dir_mat_normalized, amp_wilcoxonstat_mat_normalized, amp_pval_mat_normalized, amp_pval_precise_mat_normalized, amp_median_difference_mat_normalized, amp_median_difference_lo_mat_normalized, amp_median_difference_hi_mat_normalized, amp_cohen_d_mat_normalized), f)

In [ ]:
with open('/wrk/luukinen/results/20260621/ampstagemats_normalized.pickle', 'rb') as f:
    amp_dir_mat_normalized, amp_wilcoxonstat_mat_normalized, amp_pval_mat_normalized, amp_pval_precise_mat_normalized, amp_median_difference_mat_normalized, amp_median_difference_lo_mat_normalized, amp_median_difference_hi_mat_normalized, amp_cohen_d_mat_normalized = pickle.load(f)

In [ ]:
for mat in [amp_dir_mat_normalized, amp_wilcoxonstat_mat_normalized, amp_pval_mat_normalized, amp_pval_precise_mat_normalized, amp_median_difference_mat_normalized, amp_median_difference_lo_mat_normalized, amp_median_difference_hi_mat_normalized, amp_cohen_d_mat_normalized]:
    print(mat)
    print('')

### Figures

In [ ]:
if 9690 in plt.get_fignums(): plt.close(9690)
fig= plt.figure(num=9690, figsize=(17/2.54, 6), dpi=100) # dpi=1000
gs = GridSpec(3, 2)
ax = []
ax.append(fig.add_subplot(gs[0, 0]))
ax.append(fig.add_subplot(gs[2, 0]))
ax[0].violinplot(freq_magnitudes_by_stage, showextrema=False, showmedians=True)
ax[0].set_ylim(0, np.percentile([n for l in freq_magnitudes_by_stage for n in l], 95))
ax[1].violinplot(freq_magnitudes_by_stage_normalized, showextrema=False, showmedians=True)
ax[1].set_ylim(0, np.percentile([n for l in freq_magnitudes_by_stage_normalized for n in l], 95))
for i in [0,1]:
    ax[i].set_xticks([1,2,3,4])
    plt.setp(ax[i].get_yticklabels(), **figfont10)
    ax[i].set_xticklabels(['R', 'N1', 'N2', 'N3'], **figfont10)
ax[0].set_ylabel('Freq. increase (Hz)', **figfont10)
ax[1].set_ylabel('Norm. freq. increase (a.u.)', **figfont10)
ax[0].set_xlabel('(a)', **figfont8, labelpad=8)
ax[1].set_xlabel('(d)', **figfont8, labelpad=8)

ax.append(fig.add_subplot(gs[0, 1]))
ax.append(fig.add_subplot(gs[1, 1]))
ax.append(fig.add_subplot(gs[2, 1]))
ax[2].violinplot(amp_magnitudes_by_stage, showextrema=False, showmedians=True)
ax[2].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage for n in l], 95))
ax[3].violinplot(amp_magnitudes_by_stage_relative, showextrema=False, showmedians=True)
ax[3].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage_relative for n in l], 95))
ax[4].violinplot(amp_magnitudes_by_stage_normalized, showextrema=False, showmedians=True)
ax[4].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage_normalized for n in l], 95))
for i in [2,3,4]:
    ax[i].set_xticks([1,2,3,4])
    plt.setp(ax[i].get_yticklabels(), **figfont10)
    ax[i].set_xticklabels(['R', 'N1', 'N2', 'N3'], **figfont10)
ax[2].set_ylabel('Amp. decrease (a.u.)', **figfont10)
ax[3].set_ylabel('Rel. amp. decrease (a.u.)', **figfont10)
ax[4].set_ylabel('Norm. amp. decrease (a.u.)', **figfont10)
ax[2].set_xlabel('(b)', **figfont8, labelpad=8)
ax[3].set_xlabel('(c)', **figfont8, labelpad=8)
ax[4].set_xlabel('(e)', **figfont8, labelpad=8)

plt.tight_layout()
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig1_revised.png')

In [ ]:
print([np.median(l) for l in freq_magnitudes_by_stage])
print([np.median(l) for l in freq_magnitudes_by_stage_normalized])

In [ ]:
print([np.median(l) for l in amp_magnitudes_by_stage])
print([np.median(l) for l in amp_magnitudes_by_stage_relative])
print([np.median(l) for l in amp_magnitudes_by_stage_normalized])

## Patientwise variation

In [ ]:
indices_by_patient = [[i for i in range(len(labels)) if labels[i][0] == patient] for patient in tqdm(patients)]
indices_by_patient_spont = [[i for i in range(len(labels_spont)) if labels_spont[i][0] == patient] for patient in tqdm(patients)]

In [ ]:
freq_magnitudes_by_patient = [[freq_magnitudes[j] for j in indices_by_patient[i] if not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_patient_spont[i] if not np.isnan(freq_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient = [[amp_magnitudes[j] for j in indices_by_patient[i] if not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_patient_spont[i] if not np.isnan(amp_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]

In [ ]:
print('ALL STAGES\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient]))

In [ ]:
freq_magnitudes_by_patient_R = [[freq_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(freq_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_R = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_R[i]) > 1]
freq_magnitudes_by_patient_R = [l for l in freq_magnitudes_by_patient_R if len(l) > 1]

amp_magnitudes_by_patient_R = [[amp_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(amp_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_R = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_R[i]) > 1]
amp_magnitudes_by_patient_R = [l for l in amp_magnitudes_by_patient_R if len(l) > 1]

In [ ]:
print('STAGE R\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R]))

In [ ]:
freq_magnitudes_by_patient_N1 = [[freq_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(freq_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N1 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N1[i]) > 1]
freq_magnitudes_by_patient_N1 = [l for l in freq_magnitudes_by_patient_N1 if len(l) > 1]

amp_magnitudes_by_patient_N1 = [[amp_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(amp_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N1 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N1[i]) > 1]
amp_magnitudes_by_patient_N1 = [l for l in amp_magnitudes_by_patient_N1 if len(l) > 1]

In [ ]:
print('STAGE N1\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1]))

In [ ]:
freq_magnitudes_by_patient_N2 = [[freq_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(freq_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N2 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N2[i]) > 1]
freq_magnitudes_by_patient_N2 = [l for l in freq_magnitudes_by_patient_N2 if len(l) > 1]

amp_magnitudes_by_patient_N2 = [[amp_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(amp_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N2 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N2[i]) > 1]
amp_magnitudes_by_patient_N2 = [l for l in amp_magnitudes_by_patient_N2 if len(l) > 1]

In [ ]:
print('STAGE N2\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2]))

In [ ]:
freq_magnitudes_by_patient_N3 = [[freq_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(freq_magnitudes[j])] + [freq_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(freq_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N3 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N3[i]) > 1]
freq_magnitudes_by_patient_N3 = [l for l in freq_magnitudes_by_patient_N3 if len(l) > 1]

amp_magnitudes_by_patient_N3 = [[amp_magnitudes[j] for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(amp_magnitudes[j])] + [amp_magnitudes_spont[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(amp_magnitudes_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N3 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N3[i]) > 1]
amp_magnitudes_by_patient_N3 = [l for l in amp_magnitudes_by_patient_N3 if len(l) > 1]

In [ ]:
print('STAGE N3\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3]))

### Relativization

In [ ]:
amp_magnitudes_by_patient_relative = [[amp_magnitudes_relative[j] for j in indices_by_patient[i] if not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_patient_spont[i] if not np.isnan(amp_magnitudes_spont_relative[j])] for i in tqdm(range(len(patients)))]

In [ ]:
amp_magnitudes_by_patient_R_relative = [[amp_magnitudes_relative[j] for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(amp_magnitudes_spont_relative[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_R_relative = [l for l in amp_magnitudes_by_patient_R_relative if len(l) > 1]

In [ ]:
amp_magnitudes_by_patient_N1_relative = [[amp_magnitudes_relative[j] for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(amp_magnitudes_spont_relative[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N1_relative = [l for l in amp_magnitudes_by_patient_N1_relative if len(l) > 1]

In [ ]:
amp_magnitudes_by_patient_N2_relative = [[amp_magnitudes_relative[j] for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(amp_magnitudes_spont_relative[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N2_relative = [l for l in amp_magnitudes_by_patient_N2_relative if len(l) > 1]

In [ ]:
amp_magnitudes_by_patient_N3_relative = [[amp_magnitudes_relative[j] for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(amp_magnitudes_relative[j])] + [amp_magnitudes_spont_relative[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(amp_magnitudes_spont_relative[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N3_relative = [l for l in amp_magnitudes_by_patient_N3_relative if len(l) > 1]

### Normalization

In [ ]:
freq_magnitudes_by_patient_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_patient[i] if not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if not np.isnan(freq_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_patient[i] if not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if not np.isnan(amp_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]

In [ ]:
print('ALL STAGES\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_normalized]))

In [ ]:
freq_magnitudes_by_patient_R_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(freq_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_R_normalized = [l for l in freq_magnitudes_by_patient_R_normalized if len(l) > 1]
amp_magnitudes_by_patient_R_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(amp_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_R_normalized = [l for l in amp_magnitudes_by_patient_R_normalized if len(l) > 1]

In [ ]:
print('STAGE R\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_normalized]))

In [ ]:
freq_magnitudes_by_patient_N1_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(freq_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N1_normalized = [l for l in freq_magnitudes_by_patient_N1_normalized if len(l) > 1]
amp_magnitudes_by_patient_N1_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(amp_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N1_normalized = [l for l in amp_magnitudes_by_patient_N1_normalized if len(l) > 1]

In [ ]:
print('STAGE N1\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_normalized]))

In [ ]:
freq_magnitudes_by_patient_N2_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(freq_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N2_normalized = [l for l in freq_magnitudes_by_patient_N2_normalized if len(l) > 1]
amp_magnitudes_by_patient_N2_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(amp_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N2_normalized = [l for l in amp_magnitudes_by_patient_N2_normalized if len(l) > 1]

In [ ]:
print('STAGE N2\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_normalized]))

In [ ]:
freq_magnitudes_by_patient_N3_normalized = [[freq_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(freq_magnitudes_normalized[j])] + [freq_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(freq_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N3_normalized = [l for l in freq_magnitudes_by_patient_N3_normalized if len(l) > 1]
amp_magnitudes_by_patient_N3_normalized = [[amp_magnitudes_normalized[j] for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(amp_magnitudes_normalized[j])] + [amp_magnitudes_spont_normalized[j] for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(amp_magnitudes_spont_normalized[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N3_normalized = [l for l in amp_magnitudes_by_patient_N3_normalized if len(l) > 1]

In [ ]:
print('STAGE N3\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_normalized]))

### Comparison between absolute and normalized

In [ ]:
freq_relative_variations = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient]) / (np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R]) / (np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1]) / (np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2]) / (np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3]) / (np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.25))
]
print(freq_relative_variations)

freq_relative_variations_normalized = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.25))
]
print(freq_relative_variations_normalized)

print(scipy.stats.wilcoxon(freq_relative_variations, freq_relative_variations_normalized))

In [ ]:
amp_relative_variations = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient]) / (np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R]) / (np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1]) / (np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2]) / (np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3]) / (np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.25))
]
print(amp_relative_variations)

amp_relative_variations_relative = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_relative]) / (np.quantile([val for l in amp_magnitudes_by_patient_relative for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_relative for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_relative]) / (np.quantile([val for l in amp_magnitudes_by_patient_R_relative for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R_relative for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_relative]) / (np.quantile([val for l in amp_magnitudes_by_patient_N1_relative for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1_relative for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_relative]) / (np.quantile([val for l in amp_magnitudes_by_patient_N2_relative for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2_relative for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_relative]) / (np.quantile([val for l in amp_magnitudes_by_patient_N3_relative for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3_relative for val in l], 0.25))
]
print(amp_relative_variations_relative)

print(scipy.stats.wilcoxon(amp_relative_variations, amp_relative_variations_relative))
amp_relative_variations_normalized = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.25))
]
print(amp_relative_variations_normalized)

print(scipy.stats.wilcoxon(amp_relative_variations, amp_relative_variations_normalized))
print(scipy.stats.wilcoxon(amp_relative_variations_relative, amp_relative_variations_normalized))

## PVT number of lapses quartile comparison

In [ ]:
median_freq_magnitude_by_patient = [np.median(l) for l in freq_magnitudes_by_patient]
median_amp_magnitude_by_patient = [np.median(l) for l in amp_magnitudes_by_patient]

In [ ]:
q1_freq_magnitude, q2_freq_magnitude, q3_freq_magnitude = np.percentile(median_freq_magnitude_by_patient, [25, 50, 75])

In [ ]:
patient_inds_in_q1_freq_magnitude = [i for i in range(len(patients)) if median_freq_magnitude_by_patient[i] < q1_freq_magnitude]
patient_inds_in_q2_freq_magnitude = [i for i in range(len(patients)) if q1_freq_magnitude <= median_freq_magnitude_by_patient[i] < q2_freq_magnitude]
patient_inds_in_q3_freq_magnitude = [i for i in range(len(patients)) if q2_freq_magnitude <= median_freq_magnitude_by_patient[i] < q3_freq_magnitude]
patient_inds_in_q4_freq_magnitude = [i for i in range(len(patients)) if median_freq_magnitude_by_patient[i] >= q3_freq_magnitude]

In [ ]:
quartile_table_document = Document()

In [ ]:
lapses_table = quartile_table_document.add_table(rows=36, cols=7)
lapses_table.cell(0, 0).text = 'Table S4. PVT number of lapses'

In [ ]:
def pformatter(cell, pvalue):
    if abs(pvalue) >= 0.0001:
        cell.text = '{:.{p}g}'.format(num, p=3)
        if abs(pvalue) < 0.9995:
            while len(cell.text.lstrip('0.')) < 3:
                cell.text += '0'
    else:
        t = '{:.{p}e}'.format(pvalue, p=2)
        cell.text = t[:t.find('e')] + 'x10'
        r = cell.paragraphs[0].add_run(t[t.find('e')+1:])
        r.font.superscript = True

In [ ]:
number_of_lapses_in_freq_magnitude_quartiles = []
number_of_lapses_in_freq_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q1_freq_magnitude])
number_of_lapses_in_freq_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q2_freq_magnitude])
number_of_lapses_in_freq_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q3_freq_magnitude])
number_of_lapses_in_freq_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q4_freq_magnitude])
lapses_table.cell(1, 0).text = 'Absolute frequency response'
for i in range(4):
    lapses_table.cell(2, i).text = 'Q' + repr(i + 1) + ' median: ' + repr(np.median(number_of_lapses_in_freq_magnitude_quartiles[i]))
lapses_table.cell(4, 0).text = 'Median difference (95% confidence interval)'
lapses_table.cell(5, 0).text = 'Mann-Whitney U statistic'
lapses_table.cell(6, 0).text = 'p-value'
lapses_table.cell(7, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    lapses_table.cell(3, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles[i]))
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles[i]) - np.median(number_of_lapses_in_freq_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(number_of_lapses_in_freq_magnitude_quartiles[i], number_of_lapses_in_freq_magnitude_quartiles[j], 0.95))
    lapses_table.cell(4, k).text = repr(np.median(number_of_lapses_in_freq_magnitude_quartiles[i]) - np.median(number_of_lapses_in_freq_magnitude_quartiles[j])) + ' ' + repr(median_difference_CI(number_of_lapses_in_freq_magnitude_quartiles[i], number_of_lapses_in_freq_magnitude_quartiles[j], 0.95))
    U, pval = scipy.stats.mannwhitneyu(number_of_lapses_in_freq_magnitude_quartiles[i], number_of_lapses_in_freq_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    lapses_table.cell(5, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(lapses_table.cell(6, k), pval)
    print('Cohen d:')
    n1 = len(number_of_lapses_in_freq_magnitude_quartiles[i])
    n2 = len(number_of_lapses_in_freq_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(number_of_lapses_in_freq_magnitude_quartiles[i], number_of_lapses_in_freq_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    lapses_table.cell(7, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
q1_amp_magnitude, q2_amp_magnitude, q3_amp_magnitude = np.percentile(median_amp_magnitude_by_patient, [25, 50, 75])

In [ ]:
patient_inds_in_q1_amp_magnitude = [i for i in range(len(patients)) if median_amp_magnitude_by_patient[i] < q1_amp_magnitude]
patient_inds_in_q2_amp_magnitude = [i for i in range(len(patients)) if q1_amp_magnitude <= median_amp_magnitude_by_patient[i] < q2_amp_magnitude]
patient_inds_in_q3_amp_magnitude = [i for i in range(len(patients)) if q2_amp_magnitude <= median_amp_magnitude_by_patient[i] < q3_amp_magnitude]
patient_inds_in_q4_amp_magnitude = [i for i in range(len(patients)) if median_amp_magnitude_by_patient[i] >= q3_amp_magnitude]

In [ ]:
number_of_lapses_in_amp_magnitude_quartiles = []
number_of_lapses_in_amp_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q1_amp_magnitude])
number_of_lapses_in_amp_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q2_amp_magnitude])
number_of_lapses_in_amp_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q3_amp_magnitude])
number_of_lapses_in_amp_magnitude_quartiles.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q4_amp_magnitude])
lapses_table.cell(15, 0).text = 'Absolute amplitude response'
for i in range(4):
    lapses_table.cell(16, i).text = 'Q' + repr(i + 1) + ' median: ' + repr(np.median(number_of_lapses_in_amp_magnitude_quartiles[i]))
lapses_table.cell(18, 0).text = 'Median difference (95% confidence interval)'
lapses_table.cell(19, 0).text = 'Mann-Whitney U statistic'
lapses_table.cell(20, 0).text = 'p-value'
lapses_table.cell(21, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    lapses_table.cell(17, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles[i]))
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles[i], number_of_lapses_in_amp_magnitude_quartiles[j], 0.95))
    lapses_table.cell(18, k).text = repr(np.median(number_of_lapses_in_amp_magnitude_quartiles[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles[j])) + ' ' + repr(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles[i], number_of_lapses_in_amp_magnitude_quartiles[j], 0.95))
    U, pval = scipy.stats.mannwhitneyu(number_of_lapses_in_amp_magnitude_quartiles[i], number_of_lapses_in_amp_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    lapses_table.cell(19, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(lapses_table.cell(20, k), pval)
    print('Cohen d:')
    n1 = len(number_of_lapses_in_amp_magnitude_quartiles[i])
    n2 = len(number_of_lapses_in_amp_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(number_of_lapses_in_amp_magnitude_quartiles[i], number_of_lapses_in_amp_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    lapses_table.cell(21, k).text = '{:.3g}'.format(d)
    print('')

### Relativization

In [ ]:
median_amp_magnitude_by_patient_relative = [np.median(l) for l in amp_magnitudes_by_patient_relative]

In [ ]:
q1_amp_magnitude_relative, q2_amp_magnitude_relative, q3_amp_magnitude_relative = np.percentile(median_amp_magnitude_by_patient_relative, [25, 50, 75])

In [ ]:
patient_inds_in_q1_amp_magnitude_relative = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_relative[i] < q1_amp_magnitude_relative]
patient_inds_in_q2_amp_magnitude_relative = [i for i in range(len(patients)) if q1_amp_magnitude_relative <= median_amp_magnitude_by_patient_relative[i] < q2_amp_magnitude_relative]
patient_inds_in_q3_amp_magnitude_relative = [i for i in range(len(patients)) if q2_amp_magnitude_relative <= median_amp_magnitude_by_patient_relative[i] < q3_amp_magnitude_relative]
patient_inds_in_q4_amp_magnitude_relative = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_relative[i] >= q3_amp_magnitude_relative]

In [ ]:
number_of_lapses_in_amp_magnitude_quartiles_relative = []
number_of_lapses_in_amp_magnitude_quartiles_relative.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_relative])
number_of_lapses_in_amp_magnitude_quartiles_relative.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q2_amp_magnitude_relative])
number_of_lapses_in_amp_magnitude_quartiles_relative.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q3_amp_magnitude_relative])
number_of_lapses_in_amp_magnitude_quartiles_relative.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_relative])
lapses_table.cell(22, 0).text = 'Relative amplitude response'
for i in range(4):
    lapses_table.cell(23, i).text = 'Q' + repr(i + 1) + ' median: ' + repr(np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[i]))
lapses_table.cell(25, 0).text = 'Median difference (95% confidence interval)'
lapses_table.cell(26, 0).text = 'Mann-Whitney U statistic'
lapses_table.cell(27, 0).text = 'p-value'
lapses_table.cell(28, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    lapses_table.cell(24, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[i]))
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[j]))
    print('Difference:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles_relative[i], number_of_lapses_in_amp_magnitude_quartiles_relative[j], 0.95))
    lapses_table.cell(25, k).text = repr(np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles_relative[j])) + ' ' + repr(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles_relative[i], number_of_lapses_in_amp_magnitude_quartiles_relative[j], 0.95))
    U, pval = scipy.stats.mannwhitneyu(number_of_lapses_in_amp_magnitude_quartiles_relative[i], number_of_lapses_in_amp_magnitude_quartiles_relative[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    lapses_table.cell(26, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(lapses_table.cell(27, k), pval)
    print('Cohen d:')
    n1 = len(number_of_lapses_in_amp_magnitude_quartiles_relative[i])
    n2 = len(number_of_lapses_in_amp_magnitude_quartiles_relative[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(number_of_lapses_in_amp_magnitude_quartiles_relative[i], number_of_lapses_in_amp_magnitude_quartiles_relative[j])), n1, n2)[1]
    print(d)
    lapses_table.cell(28, k).text = '{:.3g}'.format(d)
    print('')

### Normalization

In [ ]:
median_freq_magnitude_by_patient_normalized = [np.median(l) for l in freq_magnitudes_by_patient_normalized]
median_amp_magnitude_by_patient_normalized = [np.median(l) for l in amp_magnitudes_by_patient_normalized]

In [ ]:
q1_freq_magnitude_normalized, q2_freq_magnitude_normalized, q3_freq_magnitude_normalized = np.percentile(median_freq_magnitude_by_patient_normalized, [25, 50, 75])

In [ ]:
patient_inds_in_q1_freq_magnitude_normalized = [i for i in range(len(patients)) if median_freq_magnitude_by_patient_normalized[i] < q1_freq_magnitude_normalized]
patient_inds_in_q2_freq_magnitude_normalized = [i for i in range(len(patients)) if q1_freq_magnitude_normalized <= median_freq_magnitude_by_patient_normalized[i] < q2_freq_magnitude_normalized]
patient_inds_in_q3_freq_magnitude_normalized = [i for i in range(len(patients)) if q2_freq_magnitude_normalized <= median_freq_magnitude_by_patient_normalized[i] < q3_freq_magnitude_normalized]
patient_inds_in_q4_freq_magnitude_normalized = [i for i in range(len(patients)) if median_freq_magnitude_by_patient_normalized[i] >= q3_freq_magnitude_normalized]

In [ ]:
number_of_lapses_in_freq_magnitude_quartiles_normalized = []
number_of_lapses_in_freq_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q1_freq_magnitude_normalized])
number_of_lapses_in_freq_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q2_freq_magnitude_normalized])
number_of_lapses_in_freq_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q3_freq_magnitude_normalized])
number_of_lapses_in_freq_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q4_freq_magnitude_normalized])
lapses_table.cell(8, 0).text = 'Normalized frequency response'
for i in range(4):
    lapses_table.cell(9, i).text = 'Q' + repr(i + 1) + ' median: ' + repr(np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[i]))
lapses_table.cell(11, 0).text = 'Median difference (95% confidence interval)'
lapses_table.cell(12, 0).text = 'Mann-Whitney U statistic'
lapses_table.cell(13, 0).text = 'p-value'
lapses_table.cell(14, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    lapses_table.cell(10, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[i]))
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[i]) - np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(number_of_lapses_in_freq_magnitude_quartiles_normalized[i], number_of_lapses_in_freq_magnitude_quartiles_normalized[j], 0.95))
    lapses_table.cell(11, k).text = repr(np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[i]) - np.median(number_of_lapses_in_freq_magnitude_quartiles_normalized[j])) + ' ' + repr(median_difference_CI(number_of_lapses_in_freq_magnitude_quartiles_normalized[i], number_of_lapses_in_freq_magnitude_quartiles_normalized[j], 0.95))
    U, pval = scipy.stats.mannwhitneyu(number_of_lapses_in_freq_magnitude_quartiles_normalized[i], number_of_lapses_in_freq_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    lapses_table.cell(12, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(lapses_table.cell(13, k), pval)
    print('Cohen d:')
    n1 = len(number_of_lapses_in_freq_magnitude_quartiles_normalized[i])
    n2 = len(number_of_lapses_in_freq_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(number_of_lapses_in_freq_magnitude_quartiles_normalized[i], number_of_lapses_in_freq_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    lapses_table.cell(14, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
q1_amp_magnitude_normalized, q2_amp_magnitude_normalized, q3_amp_magnitude_normalized = np.percentile(median_amp_magnitude_by_patient_normalized, [25, 50, 75])

In [ ]:
patient_inds_in_q1_amp_magnitude_normalized = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_normalized[i] < q1_amp_magnitude_normalized]
patient_inds_in_q2_amp_magnitude_normalized = [i for i in range(len(patients)) if q1_amp_magnitude_normalized <= median_amp_magnitude_by_patient_normalized[i] < q2_amp_magnitude_normalized]
patient_inds_in_q3_amp_magnitude_normalized = [i for i in range(len(patients)) if q2_amp_magnitude_normalized <= median_amp_magnitude_by_patient_normalized[i] < q3_amp_magnitude_normalized]
patient_inds_in_q4_amp_magnitude_normalized = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_normalized[i] >= q3_amp_magnitude_normalized]

In [ ]:
number_of_lapses_in_amp_magnitude_quartiles_normalized = []
number_of_lapses_in_amp_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_normalized])
number_of_lapses_in_amp_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q2_amp_magnitude_normalized])
number_of_lapses_in_amp_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q3_amp_magnitude_normalized])
number_of_lapses_in_amp_magnitude_quartiles_normalized.append([number_of_lapses_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_normalized])
lapses_table.cell(29, 0).text = 'Normalized amplitude response'
for i in range(4):
    lapses_table.cell(30, i).text = 'Q' + repr(i + 1) + ' median: ' + repr(np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[i]))
lapses_table.cell(32, 0).text = 'Median difference (95% confidence interval)'
lapses_table.cell(33, 0).text = 'Mann-Whitney U statistic'
lapses_table.cell(34, 0).text = 'p-value'
lapses_table.cell(35, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    lapses_table.cell(31, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[i]))
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles_normalized[i], number_of_lapses_in_amp_magnitude_quartiles_normalized[j], 0.95))
    lapses_table.cell(32, k).text = repr(np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[i]) - np.median(number_of_lapses_in_amp_magnitude_quartiles_normalized[j])) + ' ' + repr(median_difference_CI(number_of_lapses_in_amp_magnitude_quartiles_normalized[i], number_of_lapses_in_amp_magnitude_quartiles_normalized[j], 0.95))
    U, pval = scipy.stats.mannwhitneyu(number_of_lapses_in_amp_magnitude_quartiles_normalized[i], number_of_lapses_in_amp_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    lapses_table.cell(33, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(lapses_table.cell(34, k), pval)
    print('Cohen d:')
    n1 = len(number_of_lapses_in_amp_magnitude_quartiles_normalized[i])
    n2 = len(number_of_lapses_in_amp_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(number_of_lapses_in_amp_magnitude_quartiles_normalized[i], number_of_lapses_in_amp_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    lapses_table.cell(35, k).text = '{:.3g}'.format(d)
    print('')

## PVT mean speed quartile comparison

In [ ]:
speed_table = quartile_table_document.add_table(rows=36, cols=7)
speed_table.cell(0, 0).text = 'PVT mean speed'

In [ ]:
speed_table.cell(0, 0).text = 'Table S5. PVT mean speed'

In [ ]:
mean_speed_in_freq_magnitude_quartiles = []
mean_speed_in_freq_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q1_freq_magnitude])
mean_speed_in_freq_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q2_freq_magnitude])
mean_speed_in_freq_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q3_freq_magnitude])
mean_speed_in_freq_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q4_freq_magnitude])
speed_table.cell(1, 0).text = 'Absolute frequency response'
for i in range(4):
    speed_table.cell(2, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(mean_speed_in_freq_magnitude_quartiles[i]))
speed_table.cell(4, 0).text = 'Median difference (95% confidence interval)'
speed_table.cell(5, 0).text = 'Mann-Whitney U statistic'
speed_table.cell(6, 0).text = 'p-value'
speed_table.cell(7, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    speed_table.cell(3, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(mean_speed_in_freq_magnitude_quartiles[i]))
    print(np.median(mean_speed_in_freq_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(mean_speed_in_freq_magnitude_quartiles[i]) - np.median(mean_speed_in_freq_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(mean_speed_in_freq_magnitude_quartiles[i], mean_speed_in_freq_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(mean_speed_in_freq_magnitude_quartiles[i], mean_speed_in_freq_magnitude_quartiles[j], 0.95)
    speed_table.cell(4, k).text = '{:.3g}'.format(np.median(mean_speed_in_freq_magnitude_quartiles[i]) - np.median(mean_speed_in_freq_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(mean_speed_in_freq_magnitude_quartiles[i], mean_speed_in_freq_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    speed_table.cell(5, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(speed_table.cell(6, k), pval)
    print('Cohen d:')
    n1 = len(mean_speed_in_freq_magnitude_quartiles[i])
    n2 = len(mean_speed_in_freq_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(mean_speed_in_freq_magnitude_quartiles[i], mean_speed_in_freq_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    speed_table.cell(7, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
mean_speed_in_amp_magnitude_quartiles = []
mean_speed_in_amp_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q1_amp_magnitude])
mean_speed_in_amp_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q2_amp_magnitude])
mean_speed_in_amp_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q3_amp_magnitude])
mean_speed_in_amp_magnitude_quartiles.append([mean_speed_by_patient[i] for i in patient_inds_in_q4_amp_magnitude])
speed_table.cell(15, 0).text = 'Absolute amplitude response'
for i in range(4):
    speed_table.cell(16, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles[i]))
speed_table.cell(18, 0).text = 'Median difference (95% confidence interval)'
speed_table.cell(19, 0).text = 'Mann-Whitney U statistic'
speed_table.cell(20, 0).text = 'p-value'
speed_table.cell(21, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    speed_table.cell(17, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles[i]))
    print(np.median(mean_speed_in_amp_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles[i]) - np.median(mean_speed_in_amp_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(mean_speed_in_amp_magnitude_quartiles[i], mean_speed_in_amp_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(mean_speed_in_amp_magnitude_quartiles[i], mean_speed_in_amp_magnitude_quartiles[j], 0.95)
    speed_table.cell(18, k).text = '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles[i]) - np.median(mean_speed_in_amp_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(mean_speed_in_amp_magnitude_quartiles[i], mean_speed_in_amp_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    speed_table.cell(19, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(speed_table.cell(20, k), pval)
    print('Cohen d:')
    n1 = len(mean_speed_in_amp_magnitude_quartiles[i])
    n2 = len(mean_speed_in_amp_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(mean_speed_in_amp_magnitude_quartiles[i], mean_speed_in_amp_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    speed_table.cell(21, k).text = '{:.3g}'.format(d)
    print('')

### Relativization

In [ ]:
mean_speed_in_amp_magnitude_quartiles_relative = []
mean_speed_in_amp_magnitude_quartiles_relative.append([mean_speed_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_relative])
mean_speed_in_amp_magnitude_quartiles_relative.append([mean_speed_by_patient[i] for i in patient_inds_in_q2_amp_magnitude_relative])
mean_speed_in_amp_magnitude_quartiles_relative.append([mean_speed_by_patient[i] for i in patient_inds_in_q3_amp_magnitude_relative])
mean_speed_in_amp_magnitude_quartiles_relative.append([mean_speed_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_relative])
speed_table.cell(22, 0).text = 'Relative amplitude response'
for i in range(4):
    speed_table.cell(23, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles_relative[i]))
speed_table.cell(25, 0).text = 'Median difference (95% confidence interval)'
speed_table.cell(26, 0).text = 'Mann-Whitney U statistic'
speed_table.cell(27, 0).text = 'p-value'
speed_table.cell(28, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    speed_table.cell(24, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles_relative[i]))
    print(np.median(mean_speed_in_amp_magnitude_quartiles_relative[j]))
    print('Difference:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles_relative[i]) - np.median(mean_speed_in_amp_magnitude_quartiles_relative[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(mean_speed_in_amp_magnitude_quartiles_relative[i], mean_speed_in_amp_magnitude_quartiles_relative[j], 0.95))
    conf = median_difference_CI(mean_speed_in_amp_magnitude_quartiles_relative[i], mean_speed_in_amp_magnitude_quartiles_relative[j], 0.95)
    speed_table.cell(25, k).text = '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles_relative[i]) - np.median(mean_speed_in_amp_magnitude_quartiles_relative[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(mean_speed_in_amp_magnitude_quartiles_relative[i], mean_speed_in_amp_magnitude_quartiles_relative[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    speed_table.cell(26, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(speed_table.cell(27, k), pval)
    print('Cohen d:')
    n1 = len(mean_speed_in_amp_magnitude_quartiles_relative[i])
    n2 = len(mean_speed_in_amp_magnitude_quartiles_relative[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(mean_speed_in_amp_magnitude_quartiles_relative[i], mean_speed_in_amp_magnitude_quartiles_relative[j])), n1, n2)[1]
    print(d)
    speed_table.cell(28, k).text = '{:.3g}'.format(d)
    print('')

### Normalization

In [ ]:
mean_speed_in_freq_magnitude_quartiles_normalized = []
mean_speed_in_freq_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q1_freq_magnitude_normalized])
mean_speed_in_freq_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q2_freq_magnitude_normalized])
mean_speed_in_freq_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q3_freq_magnitude_normalized])
mean_speed_in_freq_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q4_freq_magnitude_normalized])
speed_table.cell(8, 0).text = 'Normalized frequency response'
for i in range(4):
    speed_table.cell(9, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(mean_speed_in_freq_magnitude_quartiles_normalized[i]))
speed_table.cell(11, 0).text = 'Median difference (95% confidence interval)'
speed_table.cell(12, 0).text = 'Mann-Whitney U statistic'
speed_table.cell(13, 0).text = 'p-value'
speed_table.cell(14, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    speed_table.cell(10, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(mean_speed_in_freq_magnitude_quartiles_normalized[i]))
    print(np.median(mean_speed_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(mean_speed_in_freq_magnitude_quartiles_normalized[i]) - np.median(mean_speed_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(mean_speed_in_freq_magnitude_quartiles_normalized[i], mean_speed_in_freq_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(mean_speed_in_freq_magnitude_quartiles_normalized[i], mean_speed_in_freq_magnitude_quartiles_normalized[j], 0.95)
    speed_table.cell(11, k).text = '{:.3g}'.format(np.median(mean_speed_in_freq_magnitude_quartiles_normalized[i]) - np.median(mean_speed_in_freq_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'

    U, pval = scipy.stats.mannwhitneyu(mean_speed_in_freq_magnitude_quartiles_normalized[i], mean_speed_in_freq_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    speed_table.cell(12, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(speed_table.cell(13, k), pval)
    print('Cohen d:')
    n1 = len(mean_speed_in_freq_magnitude_quartiles_normalized[i])
    n2 = len(mean_speed_in_freq_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(mean_speed_in_freq_magnitude_quartiles_normalized[i], mean_speed_in_freq_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    speed_table.cell(14, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
mean_speed_in_amp_magnitude_quartiles_normalized = []
mean_speed_in_amp_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_normalized])
mean_speed_in_amp_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q2_amp_magnitude_normalized])
mean_speed_in_amp_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q3_amp_magnitude_normalized])
mean_speed_in_amp_magnitude_quartiles_normalized.append([mean_speed_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_normalized])
speed_table.cell(29, 0).text = 'Normalized amplitude response'
for i in range(4):
    speed_table.cell(30, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles_normalized[i]))
speed_table.cell(32, 0).text = 'Median difference (95% confidence interval)'
speed_table.cell(33, 0).text = 'Mann-Whitney U statistic'
speed_table.cell(34, 0).text = 'p-value'
speed_table.cell(35, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    speed_table.cell(31, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles_normalized[i]))
    print(np.median(mean_speed_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(mean_speed_in_amp_magnitude_quartiles_normalized[i]) - np.median(mean_speed_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(mean_speed_in_amp_magnitude_quartiles_normalized[i], mean_speed_in_amp_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(mean_speed_in_amp_magnitude_quartiles_normalized[i], mean_speed_in_amp_magnitude_quartiles_normalized[j], 0.95)
    speed_table.cell(32, k).text = '{:.3g}'.format(np.median(mean_speed_in_amp_magnitude_quartiles_normalized[i]) - np.median(mean_speed_in_amp_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(mean_speed_in_amp_magnitude_quartiles_normalized[i], mean_speed_in_amp_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    speed_table.cell(33, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(speed_table.cell(34, k), pval)
    print('Cohen d:')
    n1 = len(mean_speed_in_amp_magnitude_quartiles_normalized[i])
    n2 = len(mean_speed_in_amp_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(mean_speed_in_amp_magnitude_quartiles_normalized[i], mean_speed_in_amp_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    speed_table.cell(35, k).text = '{:.3g}'.format(d)
    print('')

## AHI quartile comparison

In [ ]:
ahi_table = quartile_table_document.add_table(rows=36, cols=7)
ahi_table.cell(0, 0).text = 'Table S2. AHI'

In [ ]:
ahi_in_freq_magnitude_quartiles = []
ahi_in_freq_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q1_freq_magnitude])
ahi_in_freq_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q2_freq_magnitude])
ahi_in_freq_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q3_freq_magnitude])
ahi_in_freq_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q4_freq_magnitude])
ahi_table.cell(1, 0).text = 'Absolute frequency response'
for i in range(4):
    ahi_table.cell(2, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(ahi_in_freq_magnitude_quartiles[i]))
ahi_table.cell(4, 0).text = 'Median difference (95% confidence interval)'
ahi_table.cell(5, 0).text = 'Mann-Whitney U statistic'
ahi_table.cell(6, 0).text = 'p-value'
ahi_table.cell(7, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    ahi_table.cell(3, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(ahi_in_freq_magnitude_quartiles[i]))
    print(np.median(ahi_in_freq_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(ahi_in_freq_magnitude_quartiles[i]) - np.median(ahi_in_freq_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(ahi_in_freq_magnitude_quartiles[i], ahi_in_freq_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(ahi_in_freq_magnitude_quartiles[i], ahi_in_freq_magnitude_quartiles[j], 0.95)
    ahi_table.cell(4, k).text = '{:.3g}'.format(np.median(ahi_in_freq_magnitude_quartiles[i]) - np.median(ahi_in_freq_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(ahi_in_freq_magnitude_quartiles[i], ahi_in_freq_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    ahi_table.cell(5, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(ahi_table.cell(6, k), pval)
    print('Cohen d:')
    n1 = len(ahi_in_freq_magnitude_quartiles[i])
    n2 = len(ahi_in_freq_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(ahi_in_freq_magnitude_quartiles[i], ahi_in_freq_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    ahi_table.cell(7, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
ahi_in_amp_magnitude_quartiles = []
ahi_in_amp_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q1_amp_magnitude])
ahi_in_amp_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q2_amp_magnitude])
ahi_in_amp_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q3_amp_magnitude])
ahi_in_amp_magnitude_quartiles.append([ahi[i] for i in patient_inds_in_q4_amp_magnitude])
ahi_table.cell(15, 0).text = 'Absolute amplitude response'
for i in range(4):
    ahi_table.cell(16, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles[i]))
ahi_table.cell(18, 0).text = 'Median difference (95% confidence interval)'
ahi_table.cell(19, 0).text = 'Mann-Whitney U statistic'
ahi_table.cell(20, 0).text = 'p-value'
ahi_table.cell(21, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    ahi_table.cell(17, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(ahi_in_amp_magnitude_quartiles[i]))
    print(np.median(ahi_in_amp_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(ahi_in_amp_magnitude_quartiles[i]) - np.median(ahi_in_amp_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(ahi_in_amp_magnitude_quartiles[i], ahi_in_amp_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(ahi_in_amp_magnitude_quartiles[i], ahi_in_amp_magnitude_quartiles[j], 0.95)
    ahi_table.cell(18, k).text = '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles[i]) - np.median(ahi_in_amp_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(ahi_in_amp_magnitude_quartiles[i], ahi_in_amp_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    ahi_table.cell(19, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(ahi_table.cell(20, k), pval)
    print('Cohen d:')
    n1 = len(ahi_in_amp_magnitude_quartiles[i])
    n2 = len(ahi_in_amp_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(ahi_in_amp_magnitude_quartiles[i], ahi_in_amp_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    ahi_table.cell(21, k).text = '{:.3g}'.format(d)
    print('')

### Relativization

In [ ]:
ahi_in_amp_magnitude_quartiles_relative = []
ahi_in_amp_magnitude_quartiles_relative.append([ahi[i] for i in patient_inds_in_q1_amp_magnitude_relative])
ahi_in_amp_magnitude_quartiles_relative.append([ahi[i] for i in patient_inds_in_q2_amp_magnitude_relative])
ahi_in_amp_magnitude_quartiles_relative.append([ahi[i] for i in patient_inds_in_q3_amp_magnitude_relative])
ahi_in_amp_magnitude_quartiles_relative.append([ahi[i] for i in patient_inds_in_q4_amp_magnitude_relative])
ahi_table.cell(22, 0).text = 'Relative amplitude response'
for i in range(4):
    ahi_table.cell(23, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles_relative[i]))
ahi_table.cell(25, 0).text = 'Median difference (95% confidence interval)'
ahi_table.cell(26, 0).text = 'Mann-Whitney U statistic'
ahi_table.cell(27, 0).text = 'p-value'
ahi_table.cell(28, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    ahi_table.cell(24, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(ahi_in_amp_magnitude_quartiles_relative[i]))
    print(np.median(ahi_in_amp_magnitude_quartiles_relative[j]))
    print('Difference:')
    print(np.median(ahi_in_amp_magnitude_quartiles_relative[i]) - np.median(ahi_in_amp_magnitude_quartiles_relative[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(ahi_in_amp_magnitude_quartiles_relative[i], ahi_in_amp_magnitude_quartiles_relative[j], 0.95))
    conf = median_difference_CI(ahi_in_amp_magnitude_quartiles_relative[i], ahi_in_amp_magnitude_quartiles_relative[j], 0.95)
    ahi_table.cell(25, k).text = '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles_relative[i]) - np.median(ahi_in_amp_magnitude_quartiles_relative[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(ahi_in_amp_magnitude_quartiles_relative[i], ahi_in_amp_magnitude_quartiles_relative[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    ahi_table.cell(26, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(ahi_table.cell(27, k), pval)
    print('Cohen d:')
    n1 = len(ahi_in_amp_magnitude_quartiles_relative[i])
    n2 = len(ahi_in_amp_magnitude_quartiles_relative[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(ahi_in_amp_magnitude_quartiles_relative[i], ahi_in_amp_magnitude_quartiles_relative[j])), n1, n2)[1]
    print(d)
    ahi_table.cell(28, k).text = '{:.3g}'.format(d)
    print('')

### Normalization

In [ ]:
ahi_in_freq_magnitude_quartiles_normalized = []
ahi_in_freq_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q1_freq_magnitude_normalized])
ahi_in_freq_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q2_freq_magnitude_normalized])
ahi_in_freq_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q3_freq_magnitude_normalized])
ahi_in_freq_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q4_freq_magnitude_normalized])
ahi_table.cell(8, 0).text = 'Normalized frequency response'
for i in range(4):
    ahi_table.cell(9, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(ahi_in_freq_magnitude_quartiles_normalized[i]))
ahi_table.cell(11, 0).text = 'Median difference (95% confidence interval)'
ahi_table.cell(12, 0).text = 'Mann-Whitney U statistic'
ahi_table.cell(13, 0).text = 'p-value'
ahi_table.cell(14, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    ahi_table.cell(10, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(ahi_in_freq_magnitude_quartiles_normalized[i]))
    print(np.median(ahi_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(ahi_in_freq_magnitude_quartiles_normalized[i]) - np.median(ahi_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(ahi_in_freq_magnitude_quartiles_normalized[i], ahi_in_freq_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(ahi_in_freq_magnitude_quartiles_normalized[i], ahi_in_freq_magnitude_quartiles_normalized[j], 0.95)
    ahi_table.cell(11, k).text = '{:.3g}'.format(np.median(ahi_in_freq_magnitude_quartiles_normalized[i]) - np.median(ahi_in_freq_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'

    U, pval = scipy.stats.mannwhitneyu(ahi_in_freq_magnitude_quartiles_normalized[i], ahi_in_freq_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    ahi_table.cell(12, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(ahi_table.cell(13, k), pval)
    print('Cohen d:')
    n1 = len(ahi_in_freq_magnitude_quartiles_normalized[i])
    n2 = len(ahi_in_freq_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(ahi_in_freq_magnitude_quartiles_normalized[i], ahi_in_freq_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    ahi_table.cell(14, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
ahi_in_amp_magnitude_quartiles_normalized = []
ahi_in_amp_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q1_amp_magnitude_normalized])
ahi_in_amp_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q2_amp_magnitude_normalized])
ahi_in_amp_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q3_amp_magnitude_normalized])
ahi_in_amp_magnitude_quartiles_normalized.append([ahi[i] for i in patient_inds_in_q4_amp_magnitude_normalized])
ahi_table.cell(29, 0).text = 'Normalized amplitude response'
for i in range(4):
    ahi_table.cell(30, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles_normalized[i]))
ahi_table.cell(32, 0).text = 'Median difference (95% confidence interval)'
ahi_table.cell(33, 0).text = 'Mann-Whitney U statistic'
ahi_table.cell(34, 0).text = 'p-value'
ahi_table.cell(35, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    ahi_table.cell(31, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(ahi_in_amp_magnitude_quartiles_normalized[i]))
    print(np.median(ahi_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(ahi_in_amp_magnitude_quartiles_normalized[i]) - np.median(ahi_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(ahi_in_amp_magnitude_quartiles_normalized[i], ahi_in_amp_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(ahi_in_amp_magnitude_quartiles_normalized[i], ahi_in_amp_magnitude_quartiles_normalized[j], 0.95)
    ahi_table.cell(32, k).text = '{:.3g}'.format(np.median(ahi_in_amp_magnitude_quartiles_normalized[i]) - np.median(ahi_in_amp_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(ahi_in_amp_magnitude_quartiles_normalized[i], ahi_in_amp_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    ahi_table.cell(33, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(ahi_table.cell(34, k), pval)
    print('Cohen d:')
    n1 = len(ahi_in_amp_magnitude_quartiles_normalized[i])
    n2 = len(ahi_in_amp_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(ahi_in_amp_magnitude_quartiles_normalized[i], ahi_in_amp_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    ahi_table.cell(35, k).text = '{:.3g}'.format(d)
    print('')

## DesSev quartile comparison

In [ ]:
dessev_table = quartile_table_document.add_table(rows=36, cols=7)
dessev_table.cell(0, 0).text = 'Table S3. DesSev'

In [ ]:
dessev_in_freq_magnitude_quartiles = []
dessev_in_freq_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q1_freq_magnitude])
dessev_in_freq_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q2_freq_magnitude])
dessev_in_freq_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q3_freq_magnitude])
dessev_in_freq_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q4_freq_magnitude])
dessev_table.cell(1, 0).text = 'Absolute frequency response'
for i in range(4):
    dessev_table.cell(2, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(dessev_in_freq_magnitude_quartiles[i]))
dessev_table.cell(4, 0).text = 'Median difference (95% confidence interval)'
dessev_table.cell(5, 0).text = 'Mann-Whitney U statistic'
dessev_table.cell(6, 0).text = 'p-value'
dessev_table.cell(7, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    dessev_table.cell(3, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(dessev_in_freq_magnitude_quartiles[i]))
    print(np.median(dessev_in_freq_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(dessev_in_freq_magnitude_quartiles[i]) - np.median(dessev_in_freq_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(dessev_in_freq_magnitude_quartiles[i], dessev_in_freq_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(dessev_in_freq_magnitude_quartiles[i], dessev_in_freq_magnitude_quartiles[j], 0.95)
    dessev_table.cell(4, k).text = '{:.3g}'.format(np.median(dessev_in_freq_magnitude_quartiles[i]) - np.median(dessev_in_freq_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(dessev_in_freq_magnitude_quartiles[i], dessev_in_freq_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    dessev_table.cell(5, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(dessev_table.cell(6, k), pval)
    print('Cohen d:')
    n1 = len(dessev_in_freq_magnitude_quartiles[i])
    n2 = len(dessev_in_freq_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(dessev_in_freq_magnitude_quartiles[i], dessev_in_freq_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    dessev_table.cell(7, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
dessev_in_amp_magnitude_quartiles = []
dessev_in_amp_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q1_amp_magnitude])
dessev_in_amp_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q2_amp_magnitude])
dessev_in_amp_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q3_amp_magnitude])
dessev_in_amp_magnitude_quartiles.append([dessevs[patients[i]] for i in patient_inds_in_q4_amp_magnitude])
dessev_table.cell(15, 0).text = 'Absolute amplitude response'
for i in range(4):
    dessev_table.cell(16, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles[i]))
dessev_table.cell(18, 0).text = 'Median difference (95% confidence interval)'
dessev_table.cell(19, 0).text = 'Mann-Whitney U statistic'
dessev_table.cell(20, 0).text = 'p-value'
dessev_table.cell(21, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    dessev_table.cell(17, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(dessev_in_amp_magnitude_quartiles[i]))
    print(np.median(dessev_in_amp_magnitude_quartiles[j]))
    print('Difference:')
    print(np.median(dessev_in_amp_magnitude_quartiles[i]) - np.median(dessev_in_amp_magnitude_quartiles[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(dessev_in_amp_magnitude_quartiles[i], dessev_in_amp_magnitude_quartiles[j], 0.95))
    conf = median_difference_CI(dessev_in_amp_magnitude_quartiles[i], dessev_in_amp_magnitude_quartiles[j], 0.95)
    dessev_table.cell(18, k).text = '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles[i]) - np.median(dessev_in_amp_magnitude_quartiles[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(dessev_in_amp_magnitude_quartiles[i], dessev_in_amp_magnitude_quartiles[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    dessev_table.cell(19, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(dessev_table.cell(20, k), pval)
    print('Cohen d:')
    n1 = len(dessev_in_amp_magnitude_quartiles[i])
    n2 = len(dessev_in_amp_magnitude_quartiles[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(dessev_in_amp_magnitude_quartiles[i], dessev_in_amp_magnitude_quartiles[j])), n1, n2)[1]
    print(d)
    dessev_table.cell(21, k).text = '{:.3g}'.format(d)
    print('')

### Relativization

In [ ]:
dessev_in_amp_magnitude_quartiles_relative = []
dessev_in_amp_magnitude_quartiles_relative.append([dessevs[patients[i]] for i in patient_inds_in_q1_amp_magnitude_relative])
dessev_in_amp_magnitude_quartiles_relative.append([dessevs[patients[i]] for i in patient_inds_in_q2_amp_magnitude_relative])
dessev_in_amp_magnitude_quartiles_relative.append([dessevs[patients[i]] for i in patient_inds_in_q3_amp_magnitude_relative])
dessev_in_amp_magnitude_quartiles_relative.append([dessevs[patients[i]] for i in patient_inds_in_q4_amp_magnitude_relative])
dessev_table.cell(22, 0).text = 'Relative amplitude response'
for i in range(4):
    dessev_table.cell(23, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles_relative[i]))
dessev_table.cell(25, 0).text = 'Median difference (95% confidence interval)'
dessev_table.cell(26, 0).text = 'Mann-Whitney U statistic'
dessev_table.cell(27, 0).text = 'p-value'
dessev_table.cell(28, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    dessev_table.cell(24, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(dessev_in_amp_magnitude_quartiles_relative[i]))
    print(np.median(dessev_in_amp_magnitude_quartiles_relative[j]))
    print('Difference:')
    print(np.median(dessev_in_amp_magnitude_quartiles_relative[i]) - np.median(dessev_in_amp_magnitude_quartiles_relative[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(dessev_in_amp_magnitude_quartiles_relative[i], dessev_in_amp_magnitude_quartiles_relative[j], 0.95))
    conf = median_difference_CI(dessev_in_amp_magnitude_quartiles_relative[i], dessev_in_amp_magnitude_quartiles_relative[j], 0.95)
    dessev_table.cell(25, k).text = '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles_relative[i]) - np.median(dessev_in_amp_magnitude_quartiles_relative[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(dessev_in_amp_magnitude_quartiles_relative[i], dessev_in_amp_magnitude_quartiles_relative[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    dessev_table.cell(26, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(dessev_table.cell(27, k), pval)
    print('Cohen d:')
    n1 = len(dessev_in_amp_magnitude_quartiles_relative[i])
    n2 = len(dessev_in_amp_magnitude_quartiles_relative[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(dessev_in_amp_magnitude_quartiles_relative[i], dessev_in_amp_magnitude_quartiles_relative[j])), n1, n2)[1]
    print(d)
    dessev_table.cell(28, k).text = '{:.3g}'.format(d)
    print('')

### Normalization

In [ ]:
dessev_in_freq_magnitude_quartiles_normalized = []
dessev_in_freq_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q1_freq_magnitude_normalized])
dessev_in_freq_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q2_freq_magnitude_normalized])
dessev_in_freq_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q3_freq_magnitude_normalized])
dessev_in_freq_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q4_freq_magnitude_normalized])
dessev_table.cell(8, 0).text = 'Normalized frequency response'
for i in range(4):
    dessev_table.cell(9, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(dessev_in_freq_magnitude_quartiles_normalized[i]))
dessev_table.cell(11, 0).text = 'Median difference (95% confidence interval)'
dessev_table.cell(12, 0).text = 'Mann-Whitney U statistic'
dessev_table.cell(13, 0).text = 'p-value'
dessev_table.cell(14, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    dessev_table.cell(10, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(dessev_in_freq_magnitude_quartiles_normalized[i]))
    print(np.median(dessev_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(dessev_in_freq_magnitude_quartiles_normalized[i]) - np.median(dessev_in_freq_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(dessev_in_freq_magnitude_quartiles_normalized[i], dessev_in_freq_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(dessev_in_freq_magnitude_quartiles_normalized[i], dessev_in_freq_magnitude_quartiles_normalized[j], 0.95)
    dessev_table.cell(11, k).text = '{:.3g}'.format(np.median(dessev_in_freq_magnitude_quartiles_normalized[i]) - np.median(dessev_in_freq_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'

    U, pval = scipy.stats.mannwhitneyu(dessev_in_freq_magnitude_quartiles_normalized[i], dessev_in_freq_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    dessev_table.cell(12, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(dessev_table.cell(13, k), pval)
    print('Cohen d:')
    n1 = len(dessev_in_freq_magnitude_quartiles_normalized[i])
    n2 = len(dessev_in_freq_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(dessev_in_freq_magnitude_quartiles_normalized[i], dessev_in_freq_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    dessev_table.cell(14, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
dessev_in_amp_magnitude_quartiles_normalized = []
dessev_in_amp_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q1_amp_magnitude_normalized])
dessev_in_amp_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q2_amp_magnitude_normalized])
dessev_in_amp_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q3_amp_magnitude_normalized])
dessev_in_amp_magnitude_quartiles_normalized.append([dessevs[patients[i]] for i in patient_inds_in_q4_amp_magnitude_normalized])
dessev_table.cell(29, 0).text = 'Normalized amplitude response'
for i in range(4):
    dessev_table.cell(30, i).text = 'Q' + repr(i + 1) + ' median: ' + '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles_normalized[i]))
dessev_table.cell(32, 0).text = 'Median difference (95% confidence interval)'
dessev_table.cell(33, 0).text = 'Mann-Whitney U statistic'
dessev_table.cell(34, 0).text = 'p-value'
dessev_table.cell(35, 0).text = 'Cohen\'s d'
k = 0
for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    k += 1
    print('Q' + repr(i + 1) + ' vs. Q' + repr(j + 1) + ':')
    dessev_table.cell(31, k).text = 'Q' + repr(i + 1) + ' vs. Q' + repr(j + 1)
    print('Medians:')
    print(np.median(dessev_in_amp_magnitude_quartiles_normalized[i]))
    print(np.median(dessev_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference:')
    print(np.median(dessev_in_amp_magnitude_quartiles_normalized[i]) - np.median(dessev_in_amp_magnitude_quartiles_normalized[j]))
    print('Difference 95% confidence interval:')
    print(median_difference_CI(dessev_in_amp_magnitude_quartiles_normalized[i], dessev_in_amp_magnitude_quartiles_normalized[j], 0.95))
    conf = median_difference_CI(dessev_in_amp_magnitude_quartiles_normalized[i], dessev_in_amp_magnitude_quartiles_normalized[j], 0.95)
    dessev_table.cell(32, k).text = '{:.3g}'.format(np.median(dessev_in_amp_magnitude_quartiles_normalized[i]) - np.median(dessev_in_amp_magnitude_quartiles_normalized[j])) + ' (' + '{:.3g}'.format(conf[0]) + ', ' + ' (' + '{:.3g}'.format(conf[1]) + ')'
    U, pval = scipy.stats.mannwhitneyu(dessev_in_amp_magnitude_quartiles_normalized[i], dessev_in_amp_magnitude_quartiles_normalized[j], method='asymptotic')
    print('Mann-Whitney U statistic:')
    print(U)
    dessev_table.cell(33, k).text = repr(U)
    print('p-value:')
    print(pval)
    pformatter(dessev_table.cell(34, k), pval)
    print('Cohen d:')
    n1 = len(dessev_in_amp_magnitude_quartiles_normalized[i])
    n2 = len(dessev_in_amp_magnitude_quartiles_normalized[j])
    d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, mwu_sumT(dessev_in_amp_magnitude_quartiles_normalized[i], dessev_in_amp_magnitude_quartiles_normalized[j])), n1, n2)[1]
    print(d)
    dessev_table.cell(35, k).text = '{:.3g}'.format(d)
    print('')

In [ ]:
quartile_table_document.save('quartile.docx')

## Quartile comparisons figure

In [ ]:
def pvaluetext(pvalue):
    if pvalue >= 0.1:
        return '$p = %.2f$' % pvalue
    elif pvalue >= 0.01:
        return '$p = %.3f$' % pvalue
    else:
        text = '$p = %.1E$' % pvalue
        text = text[:text.index('E')] + '\\mathrm{E}' + text[text.index('E')+1:]
        return text

def cohentext(U, n1, n2, sumT):
    _, d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, sumT), n1, n2)
    return '$d = %.2f$' % d

def pvalueaxtext(x, y, texty, ax):
    U, pvalue = scipy.stats.mannwhitneyu(x, y)
    text = pvaluetext(pvalue)
    ax.text(1.5, texty[0], text, va='center', ha='center', **figfont8)
    n1 = len(x)
    n2 = len(y)
    sumT = mwu_sumT(x, y)
    text = cohentext(U, n1, n2, sumT)
    ax.text(1.5, texty[1], text, va='center', ha='center', **figfont8)

In [ ]:
if 8143 in plt.get_fignums(): plt.close(8143)
fig, axs = plt.subplots(nrows=4, ncols=2, sharex=True, sharey='row', num=8143, figsize=(17/2.54, 6), dpi=100)

axs[0,0].violinplot([ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[0,1].violinplot([ahi_in_freq_magnitude_quartiles_normalized[0], ahi_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[0,0].set_ylim(0, np.percentile(ahi, 90))
texty = [0.9*np.percentile(ahi, 90), 0.8*np.percentile(ahi, 90)]
pvalueaxtext(ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3], texty, axs[0,0])
pvalueaxtext(ahi_in_freq_magnitude_quartiles_normalized[0], ahi_in_freq_magnitude_quartiles_normalized[3], texty, axs[0,1])

axs[1,0].violinplot([dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[1,1].violinplot([dessev_in_freq_magnitude_quartiles_normalized[0], dessev_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[1,0].set_ylim(0, np.percentile(list(dessevs.values()), 90))
texty = [0.9*np.percentile(list(dessevs.values()), 90), 0.8*np.percentile(list(dessevs.values()), 90)]
pvalueaxtext(dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3], texty, axs[1,0])
pvalueaxtext(dessev_in_freq_magnitude_quartiles_normalized[0], dessev_in_freq_magnitude_quartiles_normalized[3], texty, axs[1,1])

axs[2,0].violinplot([number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[2,1].violinplot([number_of_lapses_in_freq_magnitude_quartiles_normalized[0], number_of_lapses_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[2,0].set_ylim(0, np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90))
_texty = np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90)
texty = [0.9*_texty, 0.8*_texty]
pvalueaxtext(number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3], texty, axs[2,0])
pvalueaxtext(number_of_lapses_in_freq_magnitude_quartiles_normalized[0], number_of_lapses_in_freq_magnitude_quartiles_normalized[3], texty, axs[2,1])

axs[3,0].violinplot([mean_speed_in_freq_magnitude_quartiles[0], mean_speed_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[3,1].violinplot([mean_speed_in_freq_magnitude_quartiles_normalized[0], mean_speed_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[3,0].set_ylim(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10), np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90))
texty = []
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.9*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.8*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
pvalueaxtext(mean_speed_in_freq_magnitude_quartiles[0], mean_speed_in_freq_magnitude_quartiles[3], texty, axs[3,0])
pvalueaxtext(mean_speed_in_freq_magnitude_quartiles_normalized[0], mean_speed_in_freq_magnitude_quartiles_normalized[3], texty, axs[3,1])

axs[0,0].set_title('Absolute frequency \nresponse', **figfont10)
axs[0,1].set_title('Normalized frequency \nresponse', **figfont10)

axs[0,0].set_ylabel('AHI', **figfont10)
axs[1,0].set_ylabel('DesSev', **figfont10)
axs[2,0].set_ylabel('PVT lapses', **figfont10)
axs[3,0].set_ylabel('PVT mean speed', **figfont10)

axs[0,0].set_xlabel('(a)', **figfont8)
axs[0,1].set_xlabel('(b)', **figfont8)
axs[1,0].set_xlabel('(c)', **figfont8)
axs[1,1].set_xlabel('(d)', **figfont8)
axs[2,0].set_xlabel('(e)', **figfont8)
axs[2,1].set_xlabel('(f)', **figfont8)
axs[3,0].set_xlabel('(g)', **figfont8)
axs[3,1].set_xlabel('(h)', **figfont8)

axs[0,0].set_xticks([1, 2])
axs[0,0].set_xticklabels(['Q1', 'Q4'], **figfont10)

fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
if 2693 in plt.get_fignums(): plt.close(2693)
fig, axs = plt.subplots(nrows=4, ncols=3, sharex=True, sharey='row', num=2693, figsize=(17/2.54, 6), dpi=100)

axs[0,0].violinplot([ahi_in_amp_magnitude_quartiles[0], ahi_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[0,1].violinplot([ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[0,2].violinplot([ahi_in_amp_magnitude_quartiles_normalized[0], ahi_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[0,0].set_ylim(0, np.percentile(ahi, 90))
texty = [0.9*np.percentile(ahi, 90), 0.8*np.percentile(ahi, 90)]
pvalueaxtext(ahi_in_amp_magnitude_quartiles[0], ahi_in_amp_magnitude_quartiles[3], texty, axs[0,0])
pvalueaxtext(ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3], texty, axs[0,1])
pvalueaxtext(ahi_in_amp_magnitude_quartiles_normalized[0], ahi_in_amp_magnitude_quartiles_normalized[3], texty, axs[0,2])

axs[1,0].violinplot([dessev_in_amp_magnitude_quartiles[0], dessev_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[1,1].violinplot([dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[1,2].violinplot([dessev_in_amp_magnitude_quartiles_normalized[0], dessev_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[1,0].set_ylim(0, np.percentile(list(dessevs.values()), 90))
texty = [0.9*np.percentile(list(dessevs.values()), 90), 0.8*np.percentile(list(dessevs.values()), 90)]
pvalueaxtext(dessev_in_amp_magnitude_quartiles[0], dessev_in_amp_magnitude_quartiles[3], texty, axs[1,0])
pvalueaxtext(dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3], texty, axs[1,1])
pvalueaxtext(dessev_in_amp_magnitude_quartiles_normalized[0], dessev_in_amp_magnitude_quartiles_normalized[3], texty, axs[1,2])

axs[2,0].violinplot([number_of_lapses_in_amp_magnitude_quartiles[0], number_of_lapses_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[2,1].violinplot([number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[2,2].violinplot([number_of_lapses_in_amp_magnitude_quartiles_normalized[0], number_of_lapses_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[2,0].set_ylim(0, np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90))
_texty = np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90)
texty = [0.9*_texty, 0.8*_texty]
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles[0], number_of_lapses_in_amp_magnitude_quartiles[3], texty, axs[2,0])
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3], texty, axs[2,1])
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles_normalized[0], number_of_lapses_in_amp_magnitude_quartiles_normalized[3], texty, axs[2,2])

axs[3,0].violinplot([mean_speed_in_amp_magnitude_quartiles[0], mean_speed_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[3,1].violinplot([mean_speed_in_amp_magnitude_quartiles_relative[0], mean_speed_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[3,2].violinplot([mean_speed_in_amp_magnitude_quartiles_normalized[0], mean_speed_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[3,0].set_ylim(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10), np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90))
texty = []
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.9*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.8*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles[0], mean_speed_in_amp_magnitude_quartiles[3], texty, axs[3,0])
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles_relative[0], mean_speed_in_amp_magnitude_quartiles_relative[3], texty, axs[3,1])
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles_normalized[0], mean_speed_in_amp_magnitude_quartiles_normalized[3], texty, axs[3,2])

axs[0,0].set_title('Absolute amplitude \nresponse', **figfont10)
axs[0,1].set_title('Relative amplitude \nresponse', **figfont10)
axs[0,2].set_title('Normalized amplitude \nresponse', **figfont10)

axs[0,0].set_ylabel('AHI', **figfont10)
axs[1,0].set_ylabel('DesSev', **figfont10)
axs[2,0].set_ylabel('PVT lapses', **figfont10)
axs[3,0].set_ylabel('PVT mean speed', **figfont10)

axs[0,0].set_xlabel('(a)', **figfont8)
axs[0,1].set_xlabel('(b)', **figfont8)
axs[0,2].set_xlabel('(c)', **figfont8)
axs[1,0].set_xlabel('(d)', **figfont8)
axs[1,1].set_xlabel('(e)', **figfont8)
axs[1,2].set_xlabel('(f)', **figfont8)
axs[2,0].set_xlabel('(h)', **figfont8)
axs[2,1].set_xlabel('(i)', **figfont8)
axs[2,2].set_xlabel('(j)', **figfont8)
axs[3,0].set_xlabel('(k)', **figfont8)
axs[3,1].set_xlabel('(l)', **figfont8)
axs[3,2].set_xlabel('(m)', **figfont8)

axs[0,0].set_xticks([1, 2])
axs[0,0].set_xticklabels(['Q1', 'Q4'], **figfont10)

fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
if 9169 in plt.get_fignums(): plt.close(9169)
fig, axs = plt.subplots(nrows=4, ncols=5, sharex=True, sharey='row', num=9169, figsize=(17/2.54, 6), dpi=100)

axs[0,0].violinplot([ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[0,1].violinplot([ahi_in_freq_magnitude_quartiles_normalized[0], ahi_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)
axs[0,2].violinplot([ahi_in_amp_magnitude_quartiles[0], ahi_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[0,3].violinplot([ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[0,4].violinplot([ahi_in_amp_magnitude_quartiles_normalized[0], ahi_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[0,0].set_ylim(0, np.percentile(ahi, 90))
texty = [0.9*np.percentile(ahi, 90), 0.8*np.percentile(ahi, 90)]
pvalueaxtext(ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3], texty, axs[0,0])
pvalueaxtext(ahi_in_freq_magnitude_quartiles_normalized[0], ahi_in_freq_magnitude_quartiles_normalized[3], texty, axs[0,1])
pvalueaxtext(ahi_in_amp_magnitude_quartiles[0], ahi_in_amp_magnitude_quartiles[3], texty, axs[0,2])
pvalueaxtext(ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3], texty, axs[0,3])
pvalueaxtext(ahi_in_amp_magnitude_quartiles_normalized[0], ahi_in_amp_magnitude_quartiles_normalized[3], texty, axs[0,4])

axs[1,0].violinplot([dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[1,1].violinplot([dessev_in_freq_magnitude_quartiles_normalized[0], dessev_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)
axs[1,2].violinplot([dessev_in_amp_magnitude_quartiles[0], dessev_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[1,3].violinplot([dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[1,4].violinplot([dessev_in_amp_magnitude_quartiles_normalized[0], dessev_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[1,0].set_ylim(0, np.percentile(list(dessevs.values()), 90))
texty = [0.9*np.percentile(list(dessevs.values()), 90), 0.8*np.percentile(list(dessevs.values()), 90)]
pvalueaxtext(dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3], texty, axs[1,0])
pvalueaxtext(dessev_in_freq_magnitude_quartiles_normalized[0], dessev_in_freq_magnitude_quartiles_normalized[3], texty, axs[1,1])
pvalueaxtext(dessev_in_amp_magnitude_quartiles[0], dessev_in_amp_magnitude_quartiles[3], texty, axs[1,2])
pvalueaxtext(dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3], texty, axs[1,3])
pvalueaxtext(dessev_in_amp_magnitude_quartiles_normalized[0], dessev_in_amp_magnitude_quartiles_normalized[3], texty, axs[1,4])

axs[2,0].violinplot([number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[2,1].violinplot([number_of_lapses_in_freq_magnitude_quartiles_normalized[0], number_of_lapses_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)
axs[2,2].violinplot([number_of_lapses_in_amp_magnitude_quartiles[0], number_of_lapses_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[2,3].violinplot([number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[2,4].violinplot([number_of_lapses_in_amp_magnitude_quartiles_normalized[0], number_of_lapses_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[2,0].set_ylim(0, np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90))
_texty = np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90)
texty = [0.9*_texty, 0.8*_texty]
pvalueaxtext(number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3], texty, axs[2,0])
pvalueaxtext(number_of_lapses_in_freq_magnitude_quartiles_normalized[0], number_of_lapses_in_freq_magnitude_quartiles_normalized[3], texty, axs[2,1])
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles[0], number_of_lapses_in_amp_magnitude_quartiles[3], texty, axs[2,2])
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3], texty, axs[2,3])
pvalueaxtext(number_of_lapses_in_amp_magnitude_quartiles_normalized[0], number_of_lapses_in_amp_magnitude_quartiles_normalized[3], texty, axs[2,4])

axs[3,0].violinplot([mean_speed_in_freq_magnitude_quartiles[0], mean_speed_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[3,1].violinplot([mean_speed_in_freq_magnitude_quartiles_normalized[0], mean_speed_in_freq_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)
axs[3,2].violinplot([mean_speed_in_amp_magnitude_quartiles[0], mean_speed_in_amp_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[3,3].violinplot([mean_speed_in_amp_magnitude_quartiles_relative[0], mean_speed_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)
axs[3,4].violinplot([mean_speed_in_amp_magnitude_quartiles_normalized[0], mean_speed_in_amp_magnitude_quartiles_normalized[3]], showextrema=False, showmedians=True)

axs[3,0].set_ylim(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10), np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90))
texty = []
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.9*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
texty.append(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.8*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10)))
pvalueaxtext(mean_speed_in_freq_magnitude_quartiles[0], mean_speed_in_freq_magnitude_quartiles[3], texty, axs[3,0])
pvalueaxtext(mean_speed_in_freq_magnitude_quartiles_normalized[0], mean_speed_in_freq_magnitude_quartiles_normalized[3], texty, axs[3,1])
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles[0], mean_speed_in_amp_magnitude_quartiles[3], texty, axs[3,2])
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles_relative[0], mean_speed_in_amp_magnitude_quartiles_relative[3], texty, axs[3,3])
pvalueaxtext(mean_speed_in_amp_magnitude_quartiles_normalized[0], mean_speed_in_amp_magnitude_quartiles_normalized[3], texty, axs[3,4])

axs[0,0].set_title('Absolute freq. \nresponse', **figfont10)
axs[0,1].set_title('Normalized freq. \nresponse', **figfont10)
axs[0,2].set_title('Absolute ampl. \nresponse', **figfont10)
axs[0,3].set_title('Relative ampl. \nresponse', **figfont10)
axs[0,4].set_title('Normalized ampl. \nresponse', **figfont10)

axs[0,0].set_ylabel('AHI', **figfont10)
axs[1,0].set_ylabel('DesSev', **figfont10)
axs[2,0].set_ylabel('PVT lapses', **figfont10)
axs[3,0].set_ylabel('PVT mean speed', **figfont10)

axs[0,0].set_xlabel('(a)', **figfont8)
axs[0,1].set_xlabel('(b)', **figfont8)
axs[0,2].set_xlabel('(c)', **figfont8)
axs[0,3].set_xlabel('(d)', **figfont8)
axs[0,4].set_xlabel('(e)', **figfont8)
axs[1,0].set_xlabel('(f)', **figfont8)
axs[1,1].set_xlabel('(h)', **figfont8)
axs[1,2].set_xlabel('(i)', **figfont8)
axs[1,3].set_xlabel('(j)', **figfont8)
axs[1,4].set_xlabel('(k)', **figfont8)
axs[2,0].set_xlabel('(l)', **figfont8)
axs[2,1].set_xlabel('(m)', **figfont8)
axs[2,2].set_xlabel('(n)', **figfont8)
axs[2,3].set_xlabel('(o)', **figfont8)
axs[2,4].set_xlabel('(p)', **figfont8)
axs[3,0].set_xlabel('(q)', **figfont8)
axs[3,1].set_xlabel('(r)', **figfont8)
axs[3,2].set_xlabel('(s)', **figfont8)
axs[3,3].set_xlabel('(t)', **figfont8)
axs[3,4].set_xlabel('(u)', **figfont8)

axs[0,0].set_xticks([1, 2])
axs[0,0].set_xticklabels(['Q1', 'Q4'], **figfont10)

fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig2_revised.png')

## Mixed models

In [ ]:
mlm_table_document = Document()

### Magnitude

In [ ]:
data_mlm = {}

data_mlm['magnitude_freq'] = []
data_mlm['magnitude_freq_normalized'] = []
data_mlm['magnitude_amp'] = []
data_mlm['magnitude_amp_relative'] = []
data_mlm['magnitude_amp_normalized'] = []
data_mlm['stage'] = []
data_mlm['ahi'] = []
data_mlm['dessev'] = []
data_mlm['speed'] = []
data_mlm['sex'] = []
data_mlm['age'] = []
data_mlm['bmi'] = []
data_mlm['patient'] = []



for i, patient in tqdm(enumerate(patients)):
    sex = demographics_by_patient[patient][1]
    age = demographics_by_patient[patient][2]
    bmi = demographics_by_patient[patient][3]
    _ahi = (len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient])) / len([stage for stage in stages[patient] if stage != 0])*30/3600
    dessev = dessevs[patient]
    speed = np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0])
    for stage in range(4):
        if patient in freq_magnitudes_by_stage_patients[stage]:
            data_mlm['magnitude_freq'].append(np.median([freq_magnitudes_by_stage[stage][i] for i in range(len(freq_magnitudes_by_stage[stage])) if freq_magnitudes_by_stage_patients[stage][i] == patient]))
            data_mlm['magnitude_freq_normalized'].append(np.median([freq_magnitudes_by_stage_normalized[stage][i] for i in range(len(freq_magnitudes_by_stage_normalized[stage])) if freq_magnitudes_by_stage_patients_normalized[stage][i] == patient]))
            data_mlm['magnitude_amp'].append(np.median([amp_magnitudes_by_stage[stage][i] for i in range(len(amp_magnitudes_by_stage[stage])) if amp_magnitudes_by_stage_patients[stage][i] == patient]))
            data_mlm['magnitude_amp_relative'].append(np.median([amp_magnitudes_by_stage_relative[stage][i] for i in range(len(amp_magnitudes_by_stage_relative[stage])) if amp_magnitudes_by_stage_patients_relative[stage][i] == patient]))
            data_mlm['magnitude_amp_normalized'].append(np.median([amp_magnitudes_by_stage_normalized[stage][i] for i in range(len(amp_magnitudes_by_stage_normalized[stage])) if amp_magnitudes_by_stage_patients_normalized[stage][i] == patient]))
            data_mlm['stage'].append(stage)
            data_mlm['ahi'].append(_ahi)
            data_mlm['dessev'].append(dessev)
            data_mlm['speed'].append(speed)
            data_mlm['sex'].append(sex)
            data_mlm['age'].append(age)
            data_mlm['bmi'].append(bmi)
            data_mlm['patient'].append(patient)
data_mlm = pandas.DataFrame(data_mlm)

In [ ]:
mlm_freq = MixedLM.from_formula('magnitude_freq ~ ahi + dessev + speed + age + bmi + sex', groups=data_mlm['stage'], vc_formula=None, data=data_mlm)
mlm_results_freq = mlm_freq.fit()
print(mlm_results_freq.summary())

In [ ]:
mlm_freq_normalized = MixedLM.from_formula('magnitude_freq_normalized ~ ahi + dessev + speed + age + bmi + sex', groups=data_mlm['stage'], vc_formula=None, data=data_mlm)
mlm_results_freq_normalized = mlm_freq_normalized.fit()
print(mlm_results_freq_normalized.summary())

In [ ]:
mlm_amp = MixedLM.from_formula('magnitude_amp ~ ahi + dessev + speed + age + bmi + sex', groups=data_mlm['stage'], vc_formula=None, data=data_mlm)
mlm_results_amp = mlm_amp.fit()
print(mlm_results_amp.summary())

In [ ]:
mlm_amp_relative = MixedLM.from_formula('magnitude_amp_relative ~ ahi + dessev + speed + age + bmi + sex', groups=data_mlm['stage'], vc_formula=None, data=data_mlm)
mlm_results_amp_relative = mlm_amp_relative.fit()
print(mlm_results_amp_relative.summary())

In [ ]:
mlm_amp_normalized = MixedLM.from_formula('magnitude_amp_normalized ~ ahi + dessev + speed + age + bmi + sex', groups=data_mlm['stage'], vc_formula=None, data=data_mlm)
mlm_results_amp_normalized = mlm_amp_normalized.fit()
print(mlm_results_amp_normalized.summary())

In [ ]:
magnitude_mlm_table = mlm_table_document.add_table(rows=50, cols=4)

In [ ]:
for k, results in enumerate([mlm_results_freq, mlm_results_freq_normalized, mlm_results_amp, mlm_results_amp_relative, mlm_results_amp_normalized]):
    for i, numlist in enumerate([results.params, results.bse, results.tvalues, results.pvalues]):
        column = magnitude_mlm_table.columns[i].cells
        for j, num in enumerate(numlist):
            if 9.995 <= abs(num) < 99.95:
                column[10*k + j].text = '{:.{p}f}'.format(num, p=1)
            elif 0.9995 <= abs(num) < 9.995 or num == 0:
                column[10*k + j].text = '{:.{p}f}'.format(num, p=2)
            elif abs(num) >= 0.0001:
                column[10*k + j].text = '{:.{p}g}'.format(num, p=3)
                if abs(num) < 0.9995:
                    while len(column[10*k + j].text.lstrip('0.')) < 3:
                        column[10*k + j].text += '0'
            else:
                t = '{:.{p}e}'.format(num, p=2)
                column[10*k + j].text = t[:t.find('e')] + 'x10'
                r = column[10*k + j].paragraphs[0].add_run(t[t.find('e')+1:])
                r.font.superscript = True

### Variation

In [ ]:
# Confirm that one of the stagewise variation of amplitude and frequency responses is definable iff the other is.

counters = []
for stage in range(4):
    counters.append((Counter(freq_magnitudes_by_stage_patients[stage]), Counter(amp_magnitudes_by_stage_patients[stage])))

print(all([all([(counters[stage][0][patient] >= 2) == (counters[stage][1][patient] >= 2) for patient in patients if patient in freq_magnitudes_by_stage_patients[stage]]) for stage in range(4)]))

In [ ]:
data_variation_mlm = {}

data_variation_mlm['variation_freq'] = []
data_variation_mlm['variation_freq_normalized'] = []
data_variation_mlm['variation_amp'] = []
data_variation_mlm['variation_amp_relative'] = []
data_variation_mlm['variation_amp_normalized'] = []
data_variation_mlm['stage'] = []
data_variation_mlm['ahi'] = []
data_variation_mlm['dessev'] = []
data_variation_mlm['speed'] = []
data_variation_mlm['sex'] = []
data_variation_mlm['age'] = []
data_variation_mlm['bmi'] = []
data_variation_mlm['patient'] = []

def secondminusfirst(arr):
    return arr[1] - arr[0]

for patient in tqdm(patients):
    sex = demographics_by_patient[patient][1]
    age = demographics_by_patient[patient][2]
    bmi = demographics_by_patient[patient][3]
    _ahi = (len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient])) / len([stage for stage in stages[patient] if stage != 0])*30/3600
    dessev = dessevs[patient]
    speed = np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0])
    for stage in range(4):
        if patient in freq_magnitudes_by_stage_patients[stage] and counters[stage][0][patient] >= 2:
            data_variation_mlm['variation_freq'].append(secondminusfirst(np.percentile([freq_magnitudes_by_stage[stage][i] for i in range(len(freq_magnitudes_by_stage[stage])) if freq_magnitudes_by_stage_patients[stage][i] == patient], [25, 75])))
            data_variation_mlm['variation_freq_normalized'].append(secondminusfirst(np.percentile([freq_magnitudes_by_stage_normalized[stage][i] for i in range(len(freq_magnitudes_by_stage_normalized[stage])) if freq_magnitudes_by_stage_patients_normalized[stage][i] == patient], [25, 75])))
            data_variation_mlm['variation_amp'].append(secondminusfirst(np.percentile([amp_magnitudes_by_stage[stage][i] for i in range(len(amp_magnitudes_by_stage[stage])) if amp_magnitudes_by_stage_patients[stage][i] == patient], [25, 75])))
            data_variation_mlm['variation_amp_relative'].append(secondminusfirst(np.percentile([amp_magnitudes_by_stage_relative[stage][i] for i in range(len(amp_magnitudes_by_stage_relative[stage])) if amp_magnitudes_by_stage_patients_relative[stage][i] == patient], [25, 75])))
            data_variation_mlm['variation_amp_normalized'].append(secondminusfirst(np.percentile([amp_magnitudes_by_stage_normalized[stage][i] for i in range(len(amp_magnitudes_by_stage_normalized[stage])) if amp_magnitudes_by_stage_patients_normalized[stage][i] == patient], [25, 75])))
            data_variation_mlm['stage'].append(stage)
            data_variation_mlm['ahi'].append(_ahi)
            data_variation_mlm['dessev'].append(dessev)
            data_variation_mlm['speed'].append(speed)
            data_variation_mlm['sex'].append(sex)
            data_variation_mlm['age'].append(age)
            data_variation_mlm['bmi'].append(bmi)
            data_variation_mlm['patient'].append(patient)
data_variation_mlm = pandas.DataFrame(data_variation_mlm)

In [ ]:
mlm_variation_freq = MixedLM.from_formula('variation_freq ~ ahi + dessev + speed + age + bmi + sex', groups=data_variation_mlm['stage'], vc_formula=None, data=data_variation_mlm)
mlm_results_variation_freq = mlm_variation_freq.fit()
print(mlm_results_variation_freq.summary())

In [ ]:
mlm_results_variation_freq.params

In [ ]:
mlm_results_variation_freq.pvalues

In [ ]:
mlm_variation_freq_normalized = MixedLM.from_formula('variation_freq_normalized ~ ahi + dessev + speed + age + bmi + sex', groups=data_variation_mlm['stage'], vc_formula=None, data=data_variation_mlm)
mlm_results_variation_freq_normalized = mlm_variation_freq_normalized.fit()
print(mlm_results_variation_freq_normalized.summary())

In [ ]:
mlm_results_variation_freq_normalized.params

In [ ]:
mlm_results_variation_freq_normalized.pvalues

In [ ]:
mlm_variation_amp = MixedLM.from_formula('variation_amp ~ ahi + dessev + speed + age + bmi + sex', groups=data_variation_mlm['stage'], vc_formula=None, data=data_variation_mlm)
mlm_results_variation_amp = mlm_variation_amp.fit()
print(mlm_results_variation_amp.summary())

In [ ]:
mlm_results_variation_amp.params

In [ ]:
mlm_results_variation_amp.params['dessev'], mlm_results_variation_amp.params['speed'], mlm_results_variation_amp.params['age'], mlm_results_variation_amp.params['bmi'], mlm_results_variation_amp.params['sex']

In [ ]:
mlm_results_variation_amp.pvalues

In [ ]:
mlm_variation_amp_relative = MixedLM.from_formula('variation_amp_relative ~ ahi + dessev + speed + age + bmi + sex', groups=data_variation_mlm['stage'], vc_formula=None, data=data_variation_mlm)
mlm_results_variation_amp_relative = mlm_variation_amp_relative.fit()
print(mlm_results_variation_amp_relative.summary())

In [ ]:
mlm_results_variation_amp_relative.params

In [ ]:
mlm_results_variation_amp_relative.params['bmi']

In [ ]:
mlm_results_variation_amp_relative.pvalues

In [ ]:
mlm_variation_amp_normalized = MixedLM.from_formula('variation_amp_normalized ~ ahi + dessev + speed + age + bmi + sex', groups=data_variation_mlm['stage'], vc_formula=None, data=data_variation_mlm)
mlm_results_variation_amp_normalized = mlm_variation_amp_normalized.fit()
print(mlm_results_variation_amp_normalized.summary())

In [ ]:
mlm_results_variation_amp_normalized.params

In [ ]:
mlm_results_variation_amp_normalized.pvalues

In [ ]:
variation_mlm_table = mlm_table_document.add_table(rows=50, cols=4)

In [ ]:
for k, results in enumerate([mlm_results_variation_freq, mlm_results_variation_freq_normalized, mlm_results_variation_amp, mlm_results_variation_amp_relative, mlm_results_variation_amp_normalized]):
    for i, numlist in enumerate([results.params, results.bse, results.tvalues, results.pvalues]):
        column = variation_mlm_table.columns[i].cells
        for j, num in enumerate(numlist):
            if 9.995 <= abs(num) < 99.95:
                column[10*k + j].text = '{:.{p}f}'.format(num, p=1)
            elif 0.9995 <= abs(num) < 9.995 or num == 0:
                column[10*k + j].text = '{:.{p}f}'.format(num, p=2)
            elif abs(num) >= 0.0001:
                column[10*k + j].text = '{:.{p}g}'.format(num, p=3)
                if abs(num) < 0.9995:
                    while len(column[10*k + j].text.lstrip('0.')) < 3:
                        column[10*k + j].text += '0'
            else:
                t = '{:.{p}e}'.format(num, p=2)
                column[10*k + j].text = t[:t.find('e')] + 'x10'
                r = column[10*k + j].paragraphs[0].add_run(t[t.find('e')+1:])
                r.font.superscript = True

In [ ]:
mlm_table_document.save('mlm.docx')

## Receiver operating characteristic

In [ ]:
def roc_point(xdata, truecats, threshold, biggerisbetter=True):
    if biggerisbetter:
        counts = Counter(zip([x < threshold for x in xdata], truecats))
    else:
        counts = Counter(zip([x > threshold for x in xdata], truecats))
    pos = truecats.count(True)
    return counts[(True, True)] / pos, counts[(True, False)] / pos

def roc_curve(xdata, truecats, biggerisbetter=True):
    tpr = []
    fpr = []
    for threshold in sorted(xdata, key=lambda x: x*(-1) ** (not biggerisbetter)):
        t, f = roc_point(xdata, truecats, threshold, biggerisbetter=biggerisbetter)
        tpr.append(t)
        fpr.append(f)
    return tpr, fpr

In [ ]:
speed_median_female = np.median([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients if demographics_by_patient[patient][1] == 0])
speed_median_male = np.median([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients if demographics_by_patient[patient][1] == 1])
speed_labels_female = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) < speed_median_female for patient in patients if demographics_by_patient[patient][1] == 0]
speed_labels_male = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) < speed_median_male for patient in patients if demographics_by_patient[patient][1] == 1]

lapses_median_female = np.median([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients if demographics_by_patient[patient][1] == 0])
lapses_median_male = np.median([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients if demographics_by_patient[patient][1] == 1])
lapses_labels_female = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) > lapses_median_female for patient in patients if demographics_by_patient[patient][1] == 0]
lapses_labels_male = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) > lapses_median_male for patient in patients if demographics_by_patient[patient][1] == 1]

In [ ]:
amp_medians_female = [np.median(amp_magnitudes_by_patient[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 0]
amp_medians_female_relative = [np.median(amp_magnitudes_by_patient_relative[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 0]
amp_medians_female_normalized = [np.median(amp_magnitudes_by_patient_normalized[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 0]
amp_medians_male = [np.median(amp_magnitudes_by_patient[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 1]
amp_medians_male_relative = [np.median(amp_magnitudes_by_patient_relative[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 1]
amp_medians_male_normalized = [np.median(amp_magnitudes_by_patient_normalized[i]) for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 1]

ahi_female = [ahi[i] for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 0]
ahi_male = [ahi[i] for i in range(len(patients)) if demographics_by_patient[patients[i]][1] == 1]

In [ ]:
if 5879 in plt.get_fignums(): plt.close(5879)
fig, ax = plt.subplots(4, 4, sharex=True, sharey='row', num=5879, figsize=(17/2.54, 6.5), dpi=100)
ax[0, 0].plot(*roc_curve(amp_medians_female, speed_labels_female))
ax[0, 0].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female, speed_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[1, 0].plot(*roc_curve(amp_medians_female_relative, speed_labels_female))
ax[1, 0].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female_relative, speed_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[2, 0].plot(*roc_curve(amp_medians_female_normalized, speed_labels_female))
ax[2, 0].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female_normalized, speed_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[3, 0].plot(*roc_curve(ahi_female, speed_labels_female, biggerisbetter=False))
ax[3, 0].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(ahi_female, speed_labels_female, biggerisbetter=False), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[0, 1].plot(*roc_curve(amp_medians_female, lapses_labels_female))
ax[0, 1].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female, lapses_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[1, 1].plot(*roc_curve(amp_medians_female_relative, lapses_labels_female))
ax[1, 1].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female_relative, lapses_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[2, 1].plot(*roc_curve(amp_medians_female_normalized, lapses_labels_female))
ax[2, 1].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_female_normalized, lapses_labels_female), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[3, 1].plot(*roc_curve(ahi_female, lapses_labels_female, biggerisbetter=False))
ax[3, 1].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(ahi_female, lapses_labels_female, biggerisbetter=False), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[0, 2].plot(*roc_curve(amp_medians_male, speed_labels_male))
ax[0, 2].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male, speed_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[1, 2].plot(*roc_curve(amp_medians_male_relative, speed_labels_male))
ax[1, 2].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male_relative, speed_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[2, 2].plot(*roc_curve(amp_medians_male_normalized, speed_labels_male))
ax[2, 2].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male_normalized, speed_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[3, 2].plot(*roc_curve(ahi_male, speed_labels_male, biggerisbetter=False))
ax[3, 2].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(ahi_male, speed_labels_male, biggerisbetter=False), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[0, 3].plot(*roc_curve(amp_medians_male, lapses_labels_male))
ax[0, 3].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male, lapses_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[1, 3].plot(*roc_curve(amp_medians_male_relative, lapses_labels_male))
ax[1, 3].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male_relative, lapses_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[2, 3].plot(*roc_curve(amp_medians_male_normalized, lapses_labels_male))
ax[2, 3].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(amp_medians_male_normalized, lapses_labels_male), axis=0):.2f}', va='center', ha='center', **figfont8)
ax[3, 3].plot(*roc_curve(ahi_male, lapses_labels_male, biggerisbetter=False))
ax[3, 3].text(0.7, 0.25, f'AUC = {trapezoid(*roc_curve(ahi_male, lapses_labels_male, biggerisbetter=False), axis=0):.2f}', va='center', ha='center', **figfont8)

ax[0,0].set_title('1/RT, female', **figfont10)
ax[0,1].set_title('Lapses, female', **figfont10)
ax[0,2].set_title('1/RT, male', **figfont10)
ax[0,3].set_title('Lapses, male', **figfont10)

ax[0,0].set_ylabel('Median abs. ampl. decr.\nTrue positive rate', **figfont10)
ax[1,0].set_ylabel('Median rel. ampl. decr.\nTrue positive rate', **figfont10)
ax[2,0].set_ylabel('Median norm. ampl. decr.\nTrue positive rate', **figfont10)
ax[3,0].set_ylabel('AHI\nTrue positive rate', **figfont10)

ax[3,0].set_xlabel('False positive rate', **figfont10)
ax[3,1].set_xlabel('False positive rate', **figfont10)
ax[3,2].set_xlabel('False positive rate', **figfont10)
ax[3,3].set_xlabel('False positive rate', **figfont10)

for i in range(4):
    for j in range(4):
        plt.setp(ax[i, j].get_yticklabels(), **figfont10)

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig3_revised.png')

## Effects of normalization on the data statistics

In [ ]:
cutoff = np.nanpercentile(freq_magnitudes, 99.9)
freq_magnitudes_without_extreme_outliers = [fm for fm in tqdm(freq_magnitudes) if fm < cutoff]
freq_magnitudes_without_extreme_outliers_or_zeros = [fm for fm in tqdm(freq_magnitudes) if 0 < fm < cutoff]

In [ ]:
cutoff = np.nanpercentile(freq_magnitudes_normalized, 99.9)
freq_magnitudes_normalized_without_extreme_outliers = [fm for fm in tqdm(freq_magnitudes_normalized) if fm < cutoff]
freq_magnitudes_normalized_without_extreme_outliers_or_zeros = [fm for fm in tqdm(freq_magnitudes_normalized) if 0 < fm < cutoff]

In [ ]:
cutoff = np.nanpercentile(amp_magnitudes, 99.9)
amp_magnitudes_without_extreme_outliers = [fm for fm in tqdm(amp_magnitudes) if fm < cutoff]
amp_magnitudes_without_extreme_outliers_or_zeros = [fm for fm in tqdm(amp_magnitudes) if 0 < fm < cutoff]

In [ ]:
cutoff = np.nanpercentile(amp_magnitudes_relative, 99.9)
amp_magnitudes_relative_without_extreme_outliers = [fm for fm in tqdm(amp_magnitudes_relative) if fm < cutoff]
amp_magnitudes_relative_without_extreme_outliers_or_zeros = [fm for fm in tqdm(amp_magnitudes_relative) if 0 < fm < cutoff]

In [ ]:
cutoff = np.nanpercentile(amp_magnitudes_normalized, 99.9)
amp_magnitudes_normalized_without_extreme_outliers = [fm for fm in tqdm(amp_magnitudes_normalized) if fm < cutoff]
amp_magnitudes_normalized_without_extreme_outliers_or_zeros = [fm for fm in tqdm(amp_magnitudes_normalized) if 0 < fm < cutoff]

In [ ]:
if 4420 in plt.get_fignums(): plt.close(4420)
fig = plt.figure(num=4420, figsize=(17/2.54, 6), dpi=100)
gs = GridSpec(3, 2)
ax = []
ax.append(fig.add_subplot(gs[0, 0]))
ax.append(fig.add_subplot(gs[2, 0]))
ax[0].hist(freq_magnitudes_without_extreme_outliers_or_zeros, bins=200)
ax[1].hist(freq_magnitudes_normalized_without_extreme_outliers_or_zeros, bins=200)
ax.append(fig.add_subplot(gs[0, 1]))
ax.append(fig.add_subplot(gs[1, 1]))
ax.append(fig.add_subplot(gs[2, 1]))
ax[2].hist(amp_magnitudes_without_extreme_outliers_or_zeros, bins=200)
ax[3].hist(amp_magnitudes_relative_without_extreme_outliers_or_zeros, bins=200)
ax[4].hist(amp_magnitudes_normalized_without_extreme_outliers_or_zeros, bins=200)

ax[0].set_xlabel('Absolute frequency increase (Hz)', **figfont10)
ax[1].set_xlabel('Normalized frequency increase (a.u.)', **figfont10)
ax[2].set_xlabel('Absolute amplitude decrease (a.u.)', **figfont10)
ax[3].set_xlabel('Relative amplitude decrease (a.u.)', **figfont10)
ax[4].set_xlabel('Normalized amplitude decrease (a.u.)', **figfont10)
ax[0].set_ylabel('Arousals in bin', **figfont10)
ax[1].set_ylabel('Arousals in bin', **figfont10)
ax[2].set_ylabel('Arousals in bin', **figfont10)
ax[3].set_ylabel('Arousals in bin', **figfont10)
ax[4].set_ylabel('Arousals in bin', **figfont10)

plt.tight_layout()
plt.show()

In [ ]:
freq_magnitudes_without_zeros_by_patient = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(freq_magnitudes_by_patient)]
# freq_magnitudes_without_zeros_by_patient_relative = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(freq_magnitudes_by_patient_relative)]
freq_magnitudes_without_zeros_by_patient_normalized = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(freq_magnitudes_by_patient_normalized)]
amp_magnitudes_without_zeros_by_patient = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(amp_magnitudes_by_patient)]
amp_magnitudes_without_zeros_by_patient_relative = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(amp_magnitudes_by_patient_relative)]
amp_magnitudes_without_zeros_by_patient_normalized = [[magnitude for magnitude in patientlist if magnitude != 0] for patientlist in tqdm(amp_magnitudes_by_patient_normalized)]

In [ ]:
freq_magnitude_coefficient_of_variation_by_patient = [scipy.stats.variation(l, ddof=1) for l in tqdm(freq_magnitudes_without_zeros_by_patient)]
#freq_magnitude_coefficient_of_variation_by_patient_relative = [scipy.stats.variation(l, ddof=1) for l in tqdm(freq_magnitudes_without_zeros_by_patient_relative)]
freq_magnitude_coefficient_of_variation_by_patient_normalized = [scipy.stats.variation(l, ddof=1) for l in tqdm(freq_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
print(np.median(freq_magnitude_coefficient_of_variation_by_patient))
#print(np.median(freq_magnitude_coefficient_of_variation_by_patient_relative))
print(np.median(freq_magnitude_coefficient_of_variation_by_patient_normalized))

In [ ]:
#wilcoxon_freq_cov1 = scipy.stats.wilcoxon(freq_magnitude_coefficient_of_variation_by_patient, freq_magnitude_coefficient_of_variation_by_patient_relative, method='approx')
#cohen_freq_cov1 = effect_size_r_d(wilcoxon_freq_cov1.zstatistic, len(freq_magnitude_coefficient_of_variation_by_patient), len(freq_magnitude_coefficient_of_variation_by_patient_relative))[1]
#print(wilcoxon_freq_cov1)
#print(cohen_freq_cov1)

In [ ]:
wilcoxon_freq_cov2 = scipy.stats.wilcoxon(freq_magnitude_coefficient_of_variation_by_patient, freq_magnitude_coefficient_of_variation_by_patient_normalized, method='approx')
cohen_freq_cov2 = effect_size_r_d(wilcoxon_freq_cov2.zstatistic, len(freq_magnitude_coefficient_of_variation_by_patient), len(freq_magnitude_coefficient_of_variation_by_patient_normalized))[1]
print(wilcoxon_freq_cov2)
print(cohen_freq_cov2)

In [ ]:
#spearman_freq_cov1 = scipy.stats.spearmanr(freq_magnitude_coefficient_of_variation_by_patient, freq_magnitude_coefficient_of_variation_by_patient_relative)
#print(spearman_freq_cov1)

In [ ]:
spearman_freq_cov2 = scipy.stats.spearmanr(freq_magnitude_coefficient_of_variation_by_patient, freq_magnitude_coefficient_of_variation_by_patient_normalized)
print(spearman_freq_cov2)

In [ ]:
amp_magnitude_coefficient_of_variation_by_patient = [scipy.stats.variation(l, ddof=1) for l in tqdm(amp_magnitudes_without_zeros_by_patient)]
amp_magnitude_coefficient_of_variation_by_patient_relative = [scipy.stats.variation(l, ddof=1) for l in tqdm(amp_magnitudes_without_zeros_by_patient_relative)]
amp_magnitude_coefficient_of_variation_by_patient_normalized = [scipy.stats.variation(l, ddof=1) for l in tqdm(amp_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
print(np.median(amp_magnitude_coefficient_of_variation_by_patient))
print(np.median(amp_magnitude_coefficient_of_variation_by_patient_relative))
print(np.median(amp_magnitude_coefficient_of_variation_by_patient_normalized))

In [ ]:
wilcoxon_amp_cov1 = scipy.stats.wilcoxon(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_relative, method='approx')
cohen_amp_cov1 = effect_size_r_d(wilcoxon_amp_cov1.zstatistic, len(amp_magnitude_coefficient_of_variation_by_patient), len(amp_magnitude_coefficient_of_variation_by_patient_relative))[1]
print(wilcoxon_amp_cov1)
print(cohen_amp_cov1)

In [ ]:
wilcoxon_amp_cov2 = scipy.stats.wilcoxon(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_normalized, method='approx')
cohen_amp_cov2 = effect_size_r_d(wilcoxon_amp_cov2.zstatistic, len(amp_magnitude_coefficient_of_variation_by_patient), len(amp_magnitude_coefficient_of_variation_by_patient_normalized))[1]
print(wilcoxon_amp_cov2)
print(cohen_amp_cov2)

In [ ]:
spearman_amp_cov1 = scipy.stats.spearmanr(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_relative)
print(spearman_amp_cov1)

In [ ]:
spearman_amp_cov2 = scipy.stats.spearmanr(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_normalized)
print(spearman_amp_cov2)

In [ ]:
if 2083 in plt.get_fignums(): plt.close(2083)
fig, ax = plt.subplots(1, 3, num=2083, figsize=(17/2.54, 3), dpi=100)
ax[0].scatter(freq_magnitude_coefficient_of_variation_by_patient, freq_magnitude_coefficient_of_variation_by_patient_normalized)
ax[1].scatter(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_relative)
ax[2].scatter(amp_magnitude_coefficient_of_variation_by_patient, amp_magnitude_coefficient_of_variation_by_patient_normalized)
plt.tight_layout()
plt.show()

In [ ]:
freq_magnitude_skewness_by_patient = [scipy.stats.skew(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient)]
#freq_magnitude_skewness_by_patient_relative = [scipy.stats.skew(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient_relative)]
freq_magnitude_skewness_by_patient_normalized = [scipy.stats.skew(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
print(np.median(freq_magnitude_skewness_by_patient))
#print(np.median(freq_magnitude_skewness_by_patient_relative))
print(np.median(freq_magnitude_skewness_by_patient_normalized))

In [ ]:
#wilcoxon_freq_skew1 = scipy.stats.wilcoxon(freq_magnitude_skewness_by_patient, freq_magnitude_skewness_by_patient_relative, method='approx')
#cohen_freq_skew1 = effect_size_r_d(wilcoxon_freq_skew1.zstatistic, len(freq_magnitude_skewness_by_patient), len(freq_magnitude_skewness_by_patient_relative))[1]
#print(wilcoxon_freq_skew1)
#print(cohen_freq_skew1)

In [ ]:
wilcoxon_freq_skew2 = scipy.stats.wilcoxon(freq_magnitude_skewness_by_patient, freq_magnitude_skewness_by_patient_normalized, method='approx')
cohen_freq_skew2 = effect_size_r_d(wilcoxon_freq_skew2.zstatistic, len(freq_magnitude_skewness_by_patient), len(freq_magnitude_skewness_by_patient_normalized))[1]
print(wilcoxon_freq_skew2)
print(cohen_freq_skew2)

In [ ]:
#spearman_freq_skew1 = scipy.stats.spearmanr(freq_magnitude_skewness_by_patient, freq_magnitude_skewness_by_patient_relative)
#print(spearman_freq_skew1)

In [ ]:
spearman_freq_skew2 = scipy.stats.spearmanr(freq_magnitude_skewness_by_patient, freq_magnitude_skewness_by_patient_normalized)
print(spearman_freq_skew2)

In [ ]:
amp_magnitude_skewness_by_patient = [scipy.stats.skew(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient)]
amp_magnitude_skewness_by_patient_relative = [scipy.stats.skew(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient_relative)]
amp_magnitude_skewness_by_patient_normalized = [scipy.stats.skew(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
print(np.median(amp_magnitude_skewness_by_patient))
print(np.median(amp_magnitude_skewness_by_patient_relative))
print(np.median(amp_magnitude_skewness_by_patient_normalized))

In [ ]:
wilcoxon_amp_skew1 = scipy.stats.wilcoxon(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_relative, method='approx')
cohen_amp_skew1 = effect_size_r_d(wilcoxon_amp_skew1.zstatistic, len(amp_magnitude_skewness_by_patient), len(amp_magnitude_skewness_by_patient_relative))[1]
print(wilcoxon_amp_skew1)
print(cohen_amp_skew1)

In [ ]:
wilcoxon_amp_skew2 = scipy.stats.wilcoxon(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_normalized, method='approx')
cohen_amp_skew2 = effect_size_r_d(wilcoxon_amp_skew2.zstatistic, len(amp_magnitude_skewness_by_patient), len(amp_magnitude_skewness_by_patient_normalized))[1]
print(wilcoxon_amp_skew2)
print(cohen_amp_skew2)

In [ ]:
spearman_amp_skew1 = scipy.stats.spearmanr(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_relative)
print(spearman_amp_skew1)

In [ ]:
spearman_amp_skew2 = scipy.stats.spearmanr(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_normalized)
print(spearman_amp_skew2)

In [ ]:
if 3086 in plt.get_fignums(): plt.close(3086)
fig, ax = plt.subplots(1, 3, num=3086, figsize=(17/2.54, 3), dpi=100)
ax[0].scatter(freq_magnitude_skewness_by_patient, freq_magnitude_skewness_by_patient_normalized)
ax[1].scatter(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_relative)
ax[2].scatter(amp_magnitude_skewness_by_patient, amp_magnitude_skewness_by_patient_normalized)
plt.tight_layout()
plt.show()

In [ ]:
freq_magnitude_kurtosis_by_patient = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient)]
#freq_magnitude_kurtosis_by_patient_relative = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient_relative)]
freq_magnitude_kurtosis_by_patient_normalized = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(freq_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
print(np.median(freq_magnitude_kurtosis_by_patient))
#print(np.median(freq_magnitude_kurtosis_by_patient_relative))
print(np.median(freq_magnitude_kurtosis_by_patient_normalized))

In [ ]:
#wilcoxon_freq_kurtosis1 = scipy.stats.wilcoxon(freq_magnitude_kurtosis_by_patient, freq_magnitude_kurtosis_by_patient_relative, method='approx')
#cohen_freq_kurtosis1 = effect_size_r_d(wilcoxon_freq_kurtosis1.zstatistic, len(freq_magnitude_kurtosis_by_patient), len(freq_magnitude_kurtosis_by_patient_relative))[1]
#print(wilcoxon_freq_kurtosis1)
#print(cohen_freq_kurtosis1)

In [ ]:
wilcoxon_freq_kurtosis2 = scipy.stats.wilcoxon(freq_magnitude_kurtosis_by_patient, freq_magnitude_kurtosis_by_patient_normalized, method='approx')
cohen_freq_kurtosis2 = effect_size_r_d(wilcoxon_freq_kurtosis2.zstatistic, len(freq_magnitude_kurtosis_by_patient), len(freq_magnitude_kurtosis_by_patient_normalized))[1]
print(wilcoxon_freq_kurtosis2)
print(cohen_freq_kurtosis2)

In [ ]:
#spearman_freq_kurtosis1 = scipy.stats.spearmanr(freq_magnitude_kurtosis_by_patient, freq_magnitude_kurtosis_by_patient_relative)
#print(spearman_freq_kurtosis1)

In [ ]:
spearman_freq_kurtosis2 = scipy.stats.spearmanr(freq_magnitude_kurtosis_by_patient, freq_magnitude_kurtosis_by_patient_normalized)
print(spearman_freq_kurtosis2)

In [ ]:
amp_magnitude_kurtosis_by_patient = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient)]
amp_magnitude_kurtosis_by_patient_relative = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient_relative)]
amp_magnitude_kurtosis_by_patient_normalized = [scipy.stats.kurtosis(l, bias=False) for l in tqdm(amp_magnitudes_without_zeros_by_patient_normalized)]

In [ ]:
wilcoxon_amp_kurtosis1 = scipy.stats.wilcoxon(amp_magnitude_kurtosis_by_patient, amp_magnitude_kurtosis_by_patient_relative, method='approx')
cohen_amp_kurtosis1 = effect_size_r_d(wilcoxon_amp_kurtosis1.zstatistic, len(amp_magnitude_kurtosis_by_patient), len(amp_magnitude_kurtosis_by_patient_relative))[1]
print(wilcoxon_amp_kurtosis1)
print(cohen_amp_kurtosis1)

In [ ]:
wilcoxon_amp_kurtosis2 = scipy.stats.wilcoxon(amp_magnitude_kurtosis_by_patient, amp_magnitude_kurtosis_by_patient_normalized, method='approx')
cohen_amp_kurtosis2 = effect_size_r_d(wilcoxon_amp_kurtosis2.zstatistic, len(amp_magnitude_kurtosis_by_patient), len(amp_magnitude_kurtosis_by_patient_normalized))[1]
print(wilcoxon_amp_kurtosis2)
print(cohen_amp_kurtosis2)

In [ ]:
spearman_amp_kurtosis1 = scipy.stats.spearmanr(amp_magnitude_kurtosis_by_patient, amp_magnitude_kurtosis_by_patient_relative)
print(spearman_amp_kurtosis1)

In [ ]:
spearman_amp_kurtosis2 = scipy.stats.spearmanr(amp_magnitude_kurtosis_by_patient, amp_magnitude_kurtosis_by_patient_normalized)
print(spearman_amp_kurtosis2)

In [ ]:
table = [
    {'Metric': 'Amplitude', 'Property': 'Coefficient of variation', 'Absolute median': np.median(amp_magnitude_coefficient_of_variation_by_patient),  'Relative median': np.median(amp_magnitude_coefficient_of_variation_by_patient_relative),    'Wilcoxon statistic': wilcoxon_amp_cov1[0],        'Wilcoxon p': wilcoxon_amp_cov1[1],       'Cohen d': cohen_amp_cov1,       'Spearman correlation': spearman_amp_cov1[0],       'Spearman p': spearman_amp_cov1[1]},
    {'Metric': '',          'Property': 'Skewness',                 'Absolute median': np.median(amp_magnitude_skewness_by_patient),                  'Relative median': np.median(amp_magnitude_skewness_by_patient_relative),                    'Wilcoxon statistic': wilcoxon_amp_skew1[0],       'Wilcoxon p': wilcoxon_amp_skew1[1],      'Cohen d': cohen_amp_skew1,      'Spearman correlation': spearman_amp_skew1[0],      'Spearman p': spearman_amp_skew1[1]},
    {'Metric': '',          'Property': 'Kurtosis',                 'Absolute median': np.median(amp_magnitude_kurtosis_by_patient),                  'Relative median': np.median(amp_magnitude_kurtosis_by_patient_relative),                    'Wilcoxon statistic': wilcoxon_amp_kurtosis1[0],   'Wilcoxon p': wilcoxon_amp_kurtosis1[1],  'Cohen d': cohen_amp_kurtosis1,  'Spearman correlation': spearman_amp_kurtosis1[0],  'Spearman p': spearman_amp_kurtosis1[1]}
]
print(tabulate(table, headers="keys", tablefmt="rounded_outline", floatfmt=['', '', ".4f", ".4f", ".1f", 'e', ".3f", ".4f", 'e'], missingval=''))

In [ ]:
table = [
    {'Metric': 'Frequency', 'Property': 'Coefficient of variation', 'Absolute median': np.median(freq_magnitude_coefficient_of_variation_by_patient), 'Normalized median': np.median(freq_magnitude_coefficient_of_variation_by_patient_normalized), 'Wilcoxon statistic': wilcoxon_freq_cov2[0],      'Wilcoxon p': wilcoxon_freq_cov2[1],      'Cohen d': cohen_freq_cov2,      'Spearman correlation': spearman_freq_cov2[0],      'Spearman p': spearman_freq_cov2[1]},
    {'Metric': '',          'Property': 'Skewness',                 'Absolute median': np.median(freq_magnitude_skewness_by_patient),                 'Normalized median': np.median(freq_magnitude_skewness_by_patient_normalized),                 'Wilcoxon statistic': wilcoxon_freq_skew2[0],     'Wilcoxon p': wilcoxon_freq_skew2[1],     'Cohen d': cohen_freq_skew2,     'Spearman correlation': spearman_freq_skew2[0],     'Spearman p': spearman_freq_skew2[1]},
    {'Metric': '',          'Property': 'Kurtosis',                 'Absolute median': np.median(freq_magnitude_kurtosis_by_patient),                 'Normalized median': np.median(freq_magnitude_kurtosis_by_patient_normalized),                 'Wilcoxon statistic': wilcoxon_freq_kurtosis2[0], 'Wilcoxon p': wilcoxon_freq_kurtosis2[1], 'Cohen d': cohen_freq_kurtosis2, 'Spearman correlation': spearman_freq_kurtosis2[0], 'Spearman p': spearman_freq_kurtosis2[1]},
    {'Metric': '',          'Property': '',                         'Absolute median': None,                                                          'Normalized median': None,                                                                     'Wilcoxon statistic': None,                       'Wilcoxon p': None,                       'Cohen d': None,                 'Spearman correlation': None,                       'Spearman p': None},
    {'Metric': 'Amplitude', 'Property': 'Coefficient of variation', 'Absolute median': np.median(amp_magnitude_coefficient_of_variation_by_patient),  'Normalized median': np.median(amp_magnitude_coefficient_of_variation_by_patient_normalized),  'Wilcoxon statistic': wilcoxon_amp_cov2[0],       'Wilcoxon p': wilcoxon_amp_cov2[1],       'Cohen d': cohen_amp_cov2,       'Spearman correlation': spearman_amp_cov2[0],       'Spearman p': spearman_amp_cov2[1]},
    {'Metric': '',          'Property': 'Skewness',                 'Absolute median': np.median(amp_magnitude_skewness_by_patient),                  'Normalized median': np.median(amp_magnitude_skewness_by_patient_normalized),                  'Wilcoxon statistic': wilcoxon_amp_skew2[0],      'Wilcoxon p': wilcoxon_amp_skew2[1],      'Cohen d': cohen_amp_skew2,      'Spearman correlation': spearman_amp_skew2[0],      'Spearman p': spearman_amp_skew2[1]},
    {'Metric': '',          'Property': 'Kurtosis',                 'Absolute median': np.median(amp_magnitude_kurtosis_by_patient),                  'Normalized median': np.median(amp_magnitude_kurtosis_by_patient_normalized),                  'Wilcoxon statistic': wilcoxon_amp_kurtosis2[0],  'Wilcoxon p': wilcoxon_amp_kurtosis2[1],  'Cohen d': cohen_amp_kurtosis2,  'Spearman correlation': spearman_amp_kurtosis2[0],  'Spearman p': spearman_amp_kurtosis2[1]}
]
print(tabulate(table, headers="keys", tablefmt="rounded_outline", floatfmt=['', '', ".4f", ".4f", ".1f", 'e', ".3f", ".4f", 'e'], missingval=''))

In [ ]:
norm_table_document = Document()
norm_table = norm_table_document.add_table(rows=10, cols=8)
norm_table.cell(0, 0).text = 'Metric'
norm_table.cell(0, 1).text = 'Property'
norm_table.cell(0, 2).text = 'Absolute median'
norm_table.cell(0, 3).text = 'Relative/normalized median'
norm_table.cell(0, 4).text = 'Wilcoxon statistic'
norm_table.cell(0, 5).text = 'p-value'
norm_table.cell(0, 6).text = 'Cohen d'
norm_table.cell(0, 7).text = 'Spearman correlation'

norm_table.cell(1, 0).text = 'Frequency'
norm_table.cell(1, 1).text = 'Coefficient of variation'
norm_table.cell(1, 2).text = '{:.3g}'.format(np.median(freq_magnitude_coefficient_of_variation_by_patient))
norm_table.cell(1, 3).text = '{:.3g}'.format(np.median(freq_magnitude_coefficient_of_variation_by_patient_normalized))
norm_table.cell(1, 4).text = '{:.3g}'.format(wilcoxon_freq_cov2[0])
t = '{:.2e}'.format(wilcoxon_freq_cov2[1])
norm_table.cell(1, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(1, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(1, 6).text = '{:.3g}'.format(cohen_freq_cov2)
norm_table.cell(1, 7).text = '{:.3g}'.format(spearman_freq_cov2[0])

norm_table.cell(2, 0).text = '(Absolute vs. normalized)'
norm_table.cell(2, 1).text = 'Skewness'
norm_table.cell(2, 2).text = '{:.3g}'.format(np.median(freq_magnitude_skewness_by_patient))
norm_table.cell(2, 3).text = '{:.3g}'.format(np.median(freq_magnitude_skewness_by_patient_normalized))
norm_table.cell(2, 4).text = '{:.3g}'.format(wilcoxon_freq_skew2[0])
t = '{:.2e}'.format(wilcoxon_freq_skew2[1])
norm_table.cell(2, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(2, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(2, 6).text = '{:.3g}'.format(cohen_freq_skew2)
norm_table.cell(2, 7).text = '{:.3g}'.format(spearman_freq_skew2[0])

norm_table.cell(3, 1).text = 'Kurtosis'
norm_table.cell(3, 2).text = '{:.3g}'.format(np.median(freq_magnitude_kurtosis_by_patient))
norm_table.cell(3, 3).text = '{:.3g}'.format(np.median(freq_magnitude_kurtosis_by_patient_normalized))
norm_table.cell(3, 4).text = '{:.3g}'.format(wilcoxon_freq_kurtosis2[0])
t = '{:.2e}'.format(wilcoxon_freq_kurtosis2[1])
norm_table.cell(3, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(3, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(3, 6).text = '{:.3g}'.format(cohen_freq_kurtosis2)
norm_table.cell(3, 7).text = '{:.3g}'.format(spearman_freq_kurtosis2[0])

norm_table.cell(4, 0).text = 'Amplitude'
norm_table.cell(4, 1).text = 'Coefficient of variation'
norm_table.cell(4, 2).text = '{:.3g}'.format(np.median(amp_magnitude_coefficient_of_variation_by_patient))
norm_table.cell(4, 3).text = '{:.3g}'.format(np.median(amp_magnitude_coefficient_of_variation_by_patient_relative))
norm_table.cell(4, 4).text = '{:.3g}'.format(wilcoxon_amp_cov1[0])
t = '{:.2e}'.format(wilcoxon_amp_cov1[1])
norm_table.cell(4, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(4, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(4, 6).text = '{:.3g}'.format(cohen_amp_cov1)
norm_table.cell(4, 7).text = '{:.3g}'.format(spearman_amp_cov1[0])

norm_table.cell(5, 0).text = '(Absolute vs. relative)'
norm_table.cell(5, 1).text = 'Skewness'
norm_table.cell(5, 2).text = '{:.3g}'.format(np.median(amp_magnitude_skewness_by_patient))
norm_table.cell(5, 3).text = '{:.3g}'.format(np.median(amp_magnitude_skewness_by_patient_relative))
norm_table.cell(5, 4).text = '{:.3g}'.format(wilcoxon_amp_skew1[0])
t = '{:.2e}'.format(wilcoxon_amp_skew1[1])
norm_table.cell(5, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(5, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(5, 6).text = '{:.3g}'.format(cohen_amp_skew1)
norm_table.cell(5, 7).text = '{:.3g}'.format(spearman_amp_skew1[0])

norm_table.cell(6, 1).text = 'Kurtosis'
norm_table.cell(6, 2).text = '{:.3g}'.format(np.median(amp_magnitude_kurtosis_by_patient))
norm_table.cell(6, 3).text = '{:.3g}'.format(np.median(amp_magnitude_kurtosis_by_patient_relative))
norm_table.cell(6, 4).text = '{:.3g}'.format(wilcoxon_amp_kurtosis1[0])
t = '{:.2e}'.format(wilcoxon_amp_kurtosis1[1])
norm_table.cell(6, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(6, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(6, 6).text = '{:.3g}'.format(cohen_amp_kurtosis1)
norm_table.cell(6, 7).text = '{:.3g}'.format(spearman_amp_kurtosis1[0])

norm_table.cell(7, 0).text = 'Amplitude'
norm_table.cell(7, 1).text = 'Coefficient of variation'
norm_table.cell(7, 2).text = '{:.3g}'.format(np.median(amp_magnitude_coefficient_of_variation_by_patient))
norm_table.cell(7, 3).text = '{:.3g}'.format(np.median(amp_magnitude_coefficient_of_variation_by_patient_normalized))
norm_table.cell(7, 4).text = '{:.3g}'.format(wilcoxon_amp_cov2[0])
t = '{:.2e}'.format(wilcoxon_amp_cov2[1])
norm_table.cell(7, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(7, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(7, 6).text = '{:.3g}'.format(cohen_amp_cov2)
norm_table.cell(7, 7).text = '{:.3g}'.format(spearman_amp_cov2[0])

norm_table.cell(8, 0).text = '(Absolute vs. normalized)'
norm_table.cell(8, 1).text = 'Skewness'
norm_table.cell(8, 2).text = '{:.3g}'.format(np.median(amp_magnitude_skewness_by_patient))
norm_table.cell(8, 3).text = '{:.3g}'.format(np.median(amp_magnitude_skewness_by_patient_normalized))
norm_table.cell(8, 4).text = '{:.3g}'.format(wilcoxon_amp_skew2[0])
t = '{:.2e}'.format(wilcoxon_amp_skew2[1])
norm_table.cell(8, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(8, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(8, 6).text = '{:.3g}'.format(cohen_amp_skew2)
norm_table.cell(8, 7).text = '{:.3g}'.format(spearman_amp_skew2[0])

norm_table.cell(9, 1).text = 'Kurtosis'
norm_table.cell(9, 2).text = '{:.3g}'.format(np.median(amp_magnitude_kurtosis_by_patient))
norm_table.cell(9, 3).text = '{:.3g}'.format(np.median(amp_magnitude_kurtosis_by_patient_normalized))
norm_table.cell(9, 4).text = '{:.3g}'.format(wilcoxon_amp_kurtosis2[0])
t = '{:.2e}'.format(wilcoxon_amp_kurtosis2[1])
norm_table.cell(9, 5).text = t[:t.find('e')] + 'x10'
r = norm_table.cell(9, 5).paragraphs[0].add_run(t[t.find('e')+1:])
r.font.superscript = True
norm_table.cell(9, 6).text = '{:.3g}'.format(cohen_amp_kurtosis2)
norm_table.cell(9, 7).text = '{:.3g}'.format(spearman_amp_kurtosis2[0])

norm_table_document.save('norm.docx')

## Fitting model quality analysis

Moved to a separate notebook.

## Poster figs

In [ ]:
if 9690 in plt.get_fignums(): plt.close(3292)
fig= plt.figure(num=3292, figsize=(13/2.54, 5/2.54), dpi=100) # dpi=1000
gs = GridSpec(1, 2)
ax = []
ax.append(fig.add_subplot(gs[0, 0]))
ax[0].violinplot(freq_magnitudes_by_stage, showextrema=False, showmedians=True)
ax[0].set_ylim(0, np.percentile([n for l in freq_magnitudes_by_stage for n in l], 95))
ax[0].set_xticks([1,2,3,4])
ax[0].set_xticklabels(['R', 'N1', 'N2', 'N3'])
ax[0].set_ylabel('Frequency\nincrease (Hz)')
ax[0].set_xlabel('(a)', labelpad=8)

ax.append(fig.add_subplot(gs[0, 1]))
ax[1].violinplot(amp_magnitudes_by_stage_relative, showextrema=False, showmedians=True)
ax[1].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage_relative for n in l], 95))
ax[1].set_xticks([1,2,3,4])
ax[1].set_xticklabels(['R', 'N1', 'N2', 'N3'])
ax[1].set_ylabel('Amplitude\ndecrease (a.u.)')
ax[1].set_xlabel('(b)', labelpad=8)

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('posterfig1.svg')

In [ ]:
def pvaluetext(pvalue):
    if pvalue >= 0.1:
        return '$p = %.2f$' % pvalue
    elif pvalue >= 0.01:
        return '$p = %.3f$' % pvalue
    else:
        text = '$p = %.1E$' % pvalue
        text = text[:text.index('E')] + '\\mathrm{E}' + text[text.index('E')+1:]
        return text

def cohentext(U, n1, n2, sumT):
    _, d = effect_size_r_d(mannwhitneyu_to_z(U, n1, n2, sumT), n1, n2)
    return '$d = %.2f$' % d

def pvalueaxtext2(x, y, texty, ax):
    U, pvalue = scipy.stats.mannwhitneyu(x, y)
    text = pvaluetext(pvalue)
    ax.text(1.5, texty[0], text, va='center', ha='center')
    n1 = len(x)
    n2 = len(y)
    sumT = mwu_sumT(x, y)
    text = cohentext(U, n1, n2, sumT)
    ax.text(1.5, texty[1], text, va='center', ha='center')

In [ ]:
if 8544 in plt.get_fignums(): plt.close(8544)
fig, axs = plt.subplots(nrows=3, ncols=2, sharex=True, sharey='row', num=8544, figsize=(13/2.54, 10/2.54), dpi=100)

axs[0,0].violinplot([ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[0,1].violinplot([ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)

axs[0,0].set_ylim(0, np.percentile(ahi, 90))
texty = [0.85*np.percentile(ahi, 90), 0.65*np.percentile(ahi, 90)]
pvalueaxtext2(ahi_in_freq_magnitude_quartiles[0], ahi_in_freq_magnitude_quartiles[3], texty, axs[0,0])
pvalueaxtext2(ahi_in_amp_magnitude_quartiles_relative[0], ahi_in_amp_magnitude_quartiles_relative[3], texty, axs[0,1])

axs[1,0].violinplot([dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[1,1].violinplot([dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)

axs[1,0].set_ylim(0, np.percentile(list(dessevs.values()), 90))
texty = [0.85*np.percentile(list(dessevs.values()), 90), 0.65*np.percentile(list(dessevs.values()), 90)]
pvalueaxtext2(dessev_in_freq_magnitude_quartiles[0], dessev_in_freq_magnitude_quartiles[3], texty, axs[1,0])
pvalueaxtext2(dessev_in_amp_magnitude_quartiles_relative[0], dessev_in_amp_magnitude_quartiles_relative[3], texty, axs[1,1])

axs[2,0].violinplot([number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3]], showextrema=False, showmedians=True)
axs[2,1].violinplot([number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3]], showextrema=False, showmedians=True)

axs[2,0].set_ylim(0, np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90))
_texty = np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90)
texty = [0.85*_texty, 0.65*_texty]
pvalueaxtext2(number_of_lapses_in_freq_magnitude_quartiles[0], number_of_lapses_in_freq_magnitude_quartiles[3], texty, axs[2,0])
pvalueaxtext2(number_of_lapses_in_amp_magnitude_quartiles_relative[0], number_of_lapses_in_amp_magnitude_quartiles_relative[3], texty, axs[2,1])


axs[0,0].set_title('Quartiles by\nfrequency response')
axs[0,1].set_title('Quartiles by\namplitude response')

axs[0,0].set_ylabel('AHI')
axs[1,0].set_ylabel('DesSev')
axs[2,0].set_ylabel('PVT lapses')

axs[0,0].set_xlabel('(a)')
axs[0,1].set_xlabel('(b)')
axs[1,0].set_xlabel('(c)')
axs[1,1].set_xlabel('(d)')
axs[2,0].set_xlabel('(e)')
axs[2,1].set_xlabel('(f)')

axs[0,0].set_xticks([1, 2])
axs[0,0].set_xticklabels(['Q1', 'Q4'])

fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('posterfig2.svg')